In [1]:
%pip install librosa
%pip install scipy
%pip install ffmpeg
%pip install mediapipe
%pip install decord --upgrade
%pip install spikingjelly


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import scipy
import scipy.signal
import torch
import librosa
import matplotlib.pyplot as plt
import torch.nn as nn
from torch import Tensor
from collections.abc import Sequence
from functools import partial
from spikingjelly.activation_based import neuron, surrogate
from spikingjelly.activation_based import functional
from typing import Any, Callable, Optional, Union
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
from tqdm.notebook import tqdm
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
from decord import VideoReader, cpu


# Step 1: Visual - Frontend Processing

## 1. Chuyển từ 30fps sang 25fps


In [ ]:
# !ffmpeg -i "Dataset_Output\_Địa chấn_ lao động toàn cầu： Người trẻ buông chuột để làm lao động chân tay _ VTV24\00016\video.mp4" -r 25 -s 1920x1080 -vcodec libx264 -pix_fmt yuv420p -crf 23 -preset fast "Speech Reconstruction/tester.mp4" -y

## 2. Chuyển từ 48khz sang 16khz


In [ ]:
# !ffmpeg -hide_banner -loglevel error -y -i "Speech Reconstruction\tester.mp4" -c:v copy -acodec aac -ar 16000 -ac 1 "Speech Reconstruction\tester16khz.mp4"


## 3. Trích xuất Mouth ROI bằng Frame Landmaker

In [3]:
!curl -o face_landmarker_v2_with_blendshapes.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
 35  3.58M  35  1.28M   0      0  1.28M      0   00:02   00:01   00:01  1.28M
 80  3.58M  80  2.87M   0      0  1.41M      0   00:02   00:02          1.41M
100  3.58M 100  3.58M   0      0  1.47M      0   00:02   00:02          1.41M
100  3.58M 100  3.58M   0      0  1.47M      0   00:02   00:02          1.41M
100  3.58M 100  3.58M   0      0  1.47M      0   00:02   00:02          1.41M


### Face Mesh and npz

In [ ]:
# def draw_landmarks_on_image(rgb_image, detection_result):
#   face_landmarks_list = detection_result.face_landmarks
#   annotated_image = np.copy(rgb_image)

#   # Loop through the detected faces to visualize.
#   for idx in range(len(face_landmarks_list)):
#     face_landmarks = face_landmarks_list[idx]

#     # Draw the face landmarks.


#     drawing_utils.draw_landmarks(
#         image=annotated_image,
#         landmark_list=face_landmarks,
#         connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
#         landmark_drawing_spec=None,
#         connection_drawing_spec=drawing_styles.get_default_face_mesh_tesselation_style())
#     drawing_utils.draw_landmarks(
#         image=annotated_image,
#         landmark_list=face_landmarks,
#         connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
#         landmark_drawing_spec=None,
#         connection_drawing_spec=drawing_styles.get_default_face_mesh_contours_style())
#     drawing_utils.draw_landmarks(
#         image=annotated_image,
#         landmark_list=face_landmarks,
#         connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_LEFT_IRIS,
#           landmark_drawing_spec=None,
#           connection_drawing_spec=drawing_styles.get_default_face_mesh_iris_connections_style())
#     drawing_utils.draw_landmarks(
#         image=annotated_image,
#         landmark_list=face_landmarks,
#         connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_RIGHT_IRIS,
#           landmark_drawing_spec=None,
#           connection_drawing_spec=drawing_styles.get_default_face_mesh_iris_connections_style())

#   return annotated_image

# def plot_face_blendshapes_bar_graph(face_blendshapes):
#   # Extract the face blendshapes category names and scores.
#   face_blendshapes_names = [face_blendshapes_category.category_name for face_blendshapes_category in face_blendshapes]
#   face_blendshapes_scores = [face_blendshapes_category.score for face_blendshapes_category in face_blendshapes]
#   # The blendshapes are ordered in decreasing score value.
#   face_blendshapes_ranks = range(len(face_blendshapes_names))

#   fig, ax = plt.subplots(figsize=(12, 12))
#   bar = ax.barh(face_blendshapes_ranks, face_blendshapes_scores, label=[str(x) for x in face_blendshapes_ranks])
#   ax.set_yticks(face_blendshapes_ranks, face_blendshapes_names)
#   ax.invert_yaxis()

#   # Label each bar with values
#   for score, patch in zip(face_blendshapes_scores, bar.patches):
#     plt.text(patch.get_x() + patch.get_width(), patch.get_y(), f"{score:.4f}", va="top")

#   ax.set_xlabel('Score')
#   ax.set_title("Face Blendshapes")
#   plt.tight_layout()
#   plt.show()

In [ ]:
# # STEP 2: Cấu hình FaceLandmarker
# base_options = python.BaseOptions(model_asset_path='face_landmarker_v2_with_blendshapes.task')
# options = vision.FaceLandmarkerOptions(
#     base_options=base_options,
#     output_face_blendshapes=True,
#     output_facial_transformation_matrixes=True,
#     running_mode=vision.RunningMode.VIDEO, # Chế độ tối ưu cho video
#     num_faces=1)

# # Khởi tạo VideoReader
# video_path = "Speech Reconstruction/tester16khz.mp4"
# vr = VideoReader(video_path, ctx=cpu(0))
# fps = 25

# # Setup VideoWriter để vẽ thử kết quả (Bước 2.3)
# w, h = int(vr[0].shape[1]), int(vr[0].shape[0])
# out = cv2.VideoWriter('Speech Reconstruction/tester16khz_landmakers.mp4',
#                          cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

# all_results = []

# # Dùng context manager để tự động giải phóng detector
# with vision.FaceLandmarker.create_from_options(options) as detector:
#     for i in range(len(vr)):
#         frame = vr[i].asnumpy() # Decord trả về RGB

#         # Chuyển sang định dạng mp.Image
#         mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)

#         # Tính toán timestamp (ms) - 25fps ứng với 40ms/frame
#         frame_timestamp_ms = int(i * (1000 / fps))

#         # Dùng detect_for_video
#         result = detector.detect_for_video(mp_image, frame_timestamp_ms)
#         all_results.append(result)

#         annotated_frame = draw_landmarks_on_image(frame, result)
#         out.write(cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR))

# out.release()
# print(f"{len(all_results)} frames!")

### Save landmaker to npz

In [ ]:
# import numpy as np

# def save_landmarks_to_npz(all_results, output_path, source_name):
#     all_landmarks = []
#     all_blendshapes = []

#     for result in all_results:
#         if result and result.face_landmarks:
#             # Lấy 468 điểm (x, y, z)
#             current_frame = [[lm.x, lm.y, lm.z] for lm in result.face_landmarks[0]]
#             all_landmarks.append(current_frame)

#             # Lấy 52 chỉ số Blendshapes
#             if result.face_blendshapes:
#                 scores = [b.score for b in result.face_blendshapes[0]]
#                 all_blendshapes.append(scores)
#             else:
#                 all_blendshapes.append(np.zeros(52))
#         else:
#             # Frame lỗi hoặc không thấy mặt
#             all_landmarks.append(np.zeros((468, 3)))
#             all_blendshapes.append(np.zeros(52))

#     # Chuyển thành mảng NumPy
#     all_landmarks_np = np.array(all_landmarks)
#     all_blendshapes_np = np.array(all_blendshapes)

#     # Lưu file nén
#     np.savez_compressed(
#         output_path,
#         landmarks=all_landmarks_np,
#         blendshapes=all_blendshapes_np,
#         metadata={'fps': 25, 'source': source_name}
#     )
#     print(f"{len(all_landmarks_np)} frames vào {output_path}")

In [ ]:
# save_landmarks_to_npz(all_results, 'Speech Reconstruction/tester16khz_Full_Data.npz', 'Speech Reconstruction/tester16khz.mp4')

### Affine transform


In [4]:
def affine_trans(landmarks, frame, target_size=(224, 224)):
  h, w, _ = frame.shape
  left = np.array([landmarks[33][0] * w, landmarks[33][1] * h])
  right = np.array([landmarks[263][0] * w, landmarks[263][1] * h])

  dY = right[1] - left[1]
  dX = right[0] - left[0]

  angle = np.degrees(np.arctan2(dY, dX))

  theta = ((left[0] + right[0]) / 2, (left[1] + right[1]) / 2)

  distance = np.sqrt((left[0] - right[0])**2 + (left[1] - right[1])**2)
  ratio = target_size[0] * 0.3
  scale = ratio / distance

  Matrix = cv2.getRotationMatrix2D(theta, angle, scale)
  Matrix[0, 2] += (target_size[0] * 0.5) - theta[0]
  Matrix[1, 2] += (target_size[1] * 0.35) - theta[1]

  transform = cv2.warpAffine(frame, Matrix, target_size, flags=cv2.INTER_CUBIC)

  return transform

### Đọc file npz landmaker và áp dụng Affine

In [5]:
# def read_timestamp(video_path, landmark_path):
#   data = np.load(landmark_path, allow_pickle=True)
#   landmarks = data['landmarks']
#   vr = VideoReader(video_path, ctx=cpu(0))
#   out = cv2.VideoWriter('Speech Reconstruction/tester16khz_aff.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 25, (224, 224))
#   for i in range(len(vr)):
#     frame = vr[i].asnumpy()
#     landmark = landmarks[i]

#     if landmark is not None:
#       transform = affine_trans(landmark, frame)
#       out.write(cv2.cvtColor(transform, cv2.COLOR_RGB2BGR))
#     else:
#       out.write(np.zeros((h, w, 3), dtype=np.uint8))
#   out.release()


In [6]:
# read_timestamp('Speech Reconstruction/tester16khz.mp4', 'Speech Reconstruction/tester16khz_Full_Data.npz')

### Trích xuất môi


In [7]:
def lips(transform, landmark_path, size=112):
      data = np.load(landmark_path, allow_pickle=True)
      landmarks = data['landmarks']
      midX = (landmarks[61][0] + landmarks[291][0]) / 2
      midY = (landmarks[0][1] + landmarks[17][1]) / 2

      half = size // 2
      
      y1, y2 = max(0, int(midY - half)), int(midY + half)
      x1, x2 = max(0, int(midX - half)), int(midX + half)
      
      lip_roi = transform[y1:y2, x1:x2]

      # Nếu cắt bị thiếu do chạm biên, resize lại cho đủ 112x112
      if lip_roi.shape[0] != size or lip_roi.shape[1] != size:
            lip_roi = cv2.resize(lip_roi, (size, size))
      
      # if len(lip_roi.shape) == 3:
      #       lip_roi = cv2.cvtColor(lip_roi, cv2.COLOR_BGR2GRAY)

      return lip_roi

## 4. ĐỒNG BỘ STFT -> Mel-Spectrogram

In [8]:
# # ===== WAVEFORM MODE =====
# # Output: audio waveform chunks aligned to video frames
# # Mỗi video frame (25fps) ↔ 640 audio samples (16000/25)

# def get_aligned_audio(audio_path, num_video_frames, sr=16000, fps=25):
#     """Chuyển audio -> waveform chunks, mỗi chunk = 1 video frame."""
#     y, _ = librosa.load(audio_path, sr=sr)
    
#     hop = sr // fps  # 640 samples per frame
#     target_len = num_video_frames * hop
    
#     # Pad hoặc cắt audio cho khớp video
#     if len(y) > target_len:
#         y = y[:target_len]
#     else:
#         y = np.pad(y, (0, target_len - len(y)))
    
#     # Normalize về [-1, 1] (giống SIREN paper)
#     max_val = np.abs(y).max()
#     if max_val > 1e-8:
#         y = y / max_val
    
#     # Reshape thành (T_video, hop) = (T, 640)
#     audio_frames = y.reshape(num_video_frames, hop)
    
#     return audio_frames  # shape: (T_video, 640)


# STEP 2. SPIKING VISUAL ENCODER

## Direct Encoding

In [3]:

class SpikingDirectEncoder(nn.Sequential):
    def __init__(self, in_channels=1, out_channels=64) -> None:
        super().__init__()
        self.spatial_conv = nn.Conv3d(in_channels, 45, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3), bias=False)
        self.bn1 = nn.BatchNorm3d(45)
        self.lif1 = neuron.LIFNode(tau=2.0, surrogate_function=surrogate.ATan())

        self.temporal_conv = nn.Conv3d(45, out_channels, kernel_size=(5, 1, 1), stride=(1, 1, 1), padding=(2, 0, 0), bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)
        self.lif2 = neuron.LIFNode(tau=2.0, surrogate_function=surrogate.ATan())

    def forward(self, X):
        # X: (B, C, T, H, W)
        s1 = self.spatial_conv(X)
        b1 = self.bn1(s1)
        l1 = self.lif1(b1)

        s2 = self.temporal_conv(l1)
        b2 = self.bn2(s2)
        l2 = self.lif2(b2)
        return l2

## ResNet 2D Plus 1D feature extraction

In [4]:
class SpikingConv2DPlus1D(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: int, stride: int=1, padding: int=1):
        super().__init__(
          nn.Conv3d(
              in_channels=in_channels,
              out_channels=mid_channels,
              kernel_size=(1, 3, 3),
              stride=(1, stride, stride),
              padding=(0, padding, padding),
              bias=False,
          ),
          nn.BatchNorm3d(mid_channels),
          neuron.LIFNode(surrogate_function=surrogate.ATan()),
          nn.Conv3d(
              in_channels=mid_channels,
              out_channels=out_channels,
              kernel_size=(3, 1, 1),
              stride=(1, 1, 1),
              padding=(1, 0, 0),
              bias=False,
          ),
        )
    @staticmethod
    def get_stride(stride: int):
        return stride, stride, stride



In [5]:
class SpikingBasicBlock(nn.Module):
    expansion: int = 1
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        conv_builder: Callable[..., nn.Module],
        stride: int = 1,
        downsample: Optional[nn.Module] = None,
    ) -> None:
        mid_channels = (in_channels * out_channels * 3 * 3 * 3) // (in_channels * out_channels * 1 * 3 * 3 + in_channels * out_channels * 3 * 1 * 1)
        super().__init__()
        self.conv1 = nn.Sequential(
            conv_builder(in_channels, out_channels, mid_channels, stride),
            nn.BatchNorm3d(out_channels),
            neuron.LIFNode(tau=2.0, surrogate_function=surrogate.ATan())
        )
        self.conv2 = nn.Sequential(
            conv_builder(out_channels, out_channels, mid_channels),
            nn.BatchNorm3d(out_channels),
        )
        self.lif = neuron.LIFNode(tau=2.0, surrogate_function=surrogate.ATan())
        self.downsample = downsample
        self.stride = stride

    def forward(self, X: Tensor) -> Tensor:
        residual = X
        out = self.conv1(X)
        out = self.conv2(out)
        if self.downsample is not None:
            residual = self.downsample(X)
        out += residual
        out = self.lif(out)
        return out


In [6]:
from torch.utils.checkpoint import checkpoint
from spikingjelly.activation_based import functional

class VidResNet(nn.Module):
    def __init__(
        self,
        block = SpikingBasicBlock,
        conv_makers = [SpikingConv2DPlus1D] * 4,
        layers = [2, 2, 2, 2],
        zero_init_residual: bool = False,
    ):
        super().__init__()
        self.in_channels = 64
        self.stem = SpikingDirectEncoder(in_channels=1, out_channels=64)

        self.layer1 = self._make_layer(block, 64, conv_makers[0], layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, conv_makers[1], layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, conv_makers[2], layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, conv_makers[3], layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool3d((None, 1, 1))
        self.fc = nn.Linear(512 * block.expansion, 512)

        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            # SpikingBasicBlock kh?ng c? bn3; conv2[-1] l? BatchNorm3d cu?i c?a residual branch.
            for m in self.modules():
                if isinstance(m, SpikingBasicBlock):
                    nn.init.constant_(m.conv2[-1].weight, 0)

    # ★ Helper: checkpoint an toàn cho SNN
    def _snn_checkpoint(self, layer, X):
        """Gradient checkpointing có reset neuron states."""
        def _forward(x):
            functional.reset_net(layer)  # Reset v, spike của LIF/PLIF trong layer
            return layer(x)
        return checkpoint(_forward, X, use_reentrant=True)

    def forward(self, X: Tensor) -> Tensor:
        X = self.stem(X)
        # ★ Checkpoint từng block, không phải từng layer
        for block in self.layer1:
            X = self._snn_checkpoint(block, X)
        for block in self.layer2:
            X = self._snn_checkpoint(block, X)
        for block in self.layer3:
            X = self._snn_checkpoint(block, X)
        for block in self.layer4:
            X = self._snn_checkpoint(block, X)
        X = self.avgpool(X)
        X = X.squeeze(-1).squeeze(-1)
        X = X.permute(0, 2, 1)
        X = self.fc(X)
        return X

    def _make_layer(
        self,
        block: SpikingBasicBlock,
        out_channels: int,
        conv_builder: Callable[..., nn.Module],
        num_blocks: int,
        stride: int,
    ) -> nn.Sequential:
        downsample = None

        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv3d(self.in_channels, out_channels * block.expansion, kernel_size=1, stride=(1, stride, stride), bias=False),
                nn.BatchNorm3d(out_channels * block.expansion)
            )
        layers = []
        layers.append(block(self.in_channels, out_channels, conv_builder, stride, downsample))

        self.in_channels = out_channels * block.expansion
        for i in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels, conv_builder))

        return nn.Sequential(*layers)


## Spiking Transformer (S - ViT)

## Attention

In [7]:
class SpikingAttention(nn.Module):
    def __init__(self, z_dim=512, n_heads=8):
        super().__init__()
        self.n_heads = n_heads
        self.dim_head = z_dim // n_heads
        self.scale = self.dim_head ** -0.5

        self.q_linear = nn.Linear(z_dim, z_dim)
        self.k_linear = nn.Linear(z_dim, z_dim)
        self.v_linear = nn.Linear(z_dim, z_dim)

        self.q_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())
        self.k_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())
        self.v_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())

        self.attn_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())
        self.proj = nn.Linear(z_dim, z_dim)

    def forward(self, X):
        B, T, C = X.shape

        # Q, K, V
        q = self.q_linear(X)                # (B, T, C)
        k = self.k_linear(X)
        v = self.v_linear(X)

        # Đưa qua PLIF (flatten theo thời gian)
        q = q.reshape(B * T, C)
        q = self.q_plif(q)
        q = q.reshape(B, T, C)

        k = k.reshape(B * T, C)
        k = self.k_plif(k)
        k = k.reshape(B, T, C)

        v = v.reshape(B * T, C)
        v = self.v_plif(v)
        v = v.reshape(B, T, C)

        # Multi-head reshape
        q = q.reshape(B, T, self.n_heads, self.dim_head).permute(0, 2, 1, 3)  # (B, n_heads, T, dim_head)
        k = k.reshape(B, T, self.n_heads, self.dim_head).permute(0, 2, 1, 3)
        v = v.reshape(B, T, self.n_heads, self.dim_head).permute(0, 2, 1, 3)

        # Attention scores
        attn_scores = (q @ k.transpose(-2, -1)) * self.scale   # (B, n_heads, T, T)
        attn_scores = torch.softmax(attn_scores, dim=-1)
        attn_out = attn_scores @ v                             # (B, n_heads, T, dim_head)

        # Merge heads
        attn_out = attn_out.permute(0, 2, 1, 3).reshape(B, T, C)  # (B, T, C)

        # attn_plif
        attn_out = attn_out.reshape(B * T, C)
        attn_out = self.attn_plif(attn_out)
        attn_out = attn_out.reshape(B, T, C)

        # Projection
        out = self.proj(attn_out)          # (B, T, C)
        return out


## MLP

In [8]:
class SpikingMLP(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=2048, out_dim=512):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)

        self.plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())
        
        self.fc2 = nn.Linear(hidden_dim, out_dim)
        self.bn2 = nn.BatchNorm1d(out_dim)

    def forward(self, X):
        # X: (B, T, C)
        B, T, C = X.shape

        # fc1
        X = self.fc1(X)                     # (B, T, hidden_dim)
        # Chuẩn bị cho BN1 (yêu cầu (N, C, L) với L=1)
        X = X.reshape(B * T, -1)            # (B*T, hidden_dim)
        X = X.unsqueeze(-1)                 # (B*T, hidden_dim, 1)
        X = self.bn1(X)                     # BN trên kênh hidden_dim
        X = X.squeeze(-1)                   # (B*T, hidden_dim)

        # Neuron PLIF (xử lý từng vector độc lập)
        X = self.plif(X)                    # (B*T, hidden_dim)
        X = X.reshape(B, T, -1)             # (B, T, hidden_dim)

        # fc2
        X = self.fc2(X)                     # (B, T, out_dim)
        # BN2
        X = X.reshape(B * T, -1)            # (B*T, out_dim)
        X = X.unsqueeze(-1)                 # (B*T, out_dim, 1)
        X = self.bn2(X)
        X = X.squeeze(-1)                   # (B*T, out_dim)
        X = X.reshape(B, T, -1)             # (B, T, out_dim)

        return X

## SpikeTransformer

In [9]:
class SpikingVisionTransformer(nn.Module):
    def __init__(self, z_dim=512, n_heads=8, mlp_hidden_dim=2048):
        super().__init__()
        self.attn = SpikingAttention(z_dim=z_dim, n_heads=n_heads)
        self.attn_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())

        self.mlp = SpikingMLP(in_dim=z_dim, hidden_dim=mlp_hidden_dim, out_dim=z_dim)
        self.mlp_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())

    def forward(self, X):
        # X: (B, T, C)
        # Attention
        attn_out = self.attn(X)                     # (B, T, C)
        residual = attn_out + X                      # residual connection

        # attn_plif
        residual_flat = residual.reshape(-1, residual.size(-1))  # (B*T, C)
        residual_spike = self.attn_plif(residual_flat)
        residual_spike = residual_spike.reshape(residual.shape)  # (B, T, C)

        # MLP
        mlp_out = self.mlp(residual_spike)           # (B, T, C)
        residual2 = mlp_out + residual_spike          # residual connection

        # mlp_plif
        residual2_flat = residual2.reshape(-1, residual2.size(-1))  # (B*T, C)
        out_spikes = self.mlp_plif(residual2_flat)
        out_spikes = out_spikes.reshape(residual2.shape)            # (B, T, C)

        return out_spikes

## ALIF NODE

In [10]:
class ConfALIFNode(nn.Module):
    def __init__(self, tau_m=2.0, tau_a=1.5, beta=0.1):
        super().__init__()
        self.tau_m = tau_m
        self.tau_a = tau_a
        self.beta = beta
        self.surrogate = surrogate.ATan()
        self.threshold_base = 1.0

    def forward(self, X):
        B, T, C = X.shape
        v_mem = torch.zeros(B, C, device=X.device)
        a_adapt = torch.zeros(B, C, device=X.device)
        spikes_seq = []
        v_mem_seq = []
        for t in range(T):
            current_I = X[:, t, :]
            a_adapt = a_adapt * (1 - 1/self.tau_a)
            v_th = self.threshold_base + a_adapt
            v_mem = v_mem * (1 - 1/self.tau_m) + current_I
            v_mem_seq.append(v_mem)
            spike = self.surrogate(v_mem - v_th)
            v_mem = v_mem * (1 - spike)
            a_adapt = a_adapt + self.beta * spike
            spikes_seq.append(spike)
        return torch.stack(spikes_seq, dim=1), torch.stack(v_mem_seq, dim=1)


# STEP 3: Readout Layer

In [11]:
class ReadoutLayer(nn.Module):
    def __init__(self, in_dim=512, out_dim=512, tau_a=1.5, tau_m=2.0, beta=0.1):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)
        self.alif = ConfALIFNode(tau_m=tau_m, tau_a=tau_a, beta=beta)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, X): # X: B T C 
        proj_X = self.proj(X)
        _, v_mem_seq = self.alif(proj_X)
        z_continuous = self.norm(v_mem_seq)
        return z_continuous      

## SpikeTransformerEncoder

In [12]:
class SpikingViTEncoder(nn.Module):
    def __init__(self, z_dim=512, n_heads=8, n_blocks=8, mlp_hidden_dim=2048, T_max=1000):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, T_max, z_dim))
        self.backbone = VidResNet()
        self.transformers = nn.ModuleList([
            SpikingVisionTransformer(z_dim=z_dim, n_heads=n_heads, mlp_hidden_dim=mlp_hidden_dim) 
            for _ in range(n_blocks)
        ])
        self.readout = ReadoutLayer(in_dim=z_dim, out_dim=z_dim)
    
    def forward(self, X): # X: B C T H W
        features = self.backbone(X) # B T C
        features += self.pos_embedding[:, :features.size(1), :]
        for block in self.transformers:
            features = block(features) # B T C
        z_continuous = self.readout(features) # B T C
        return z_continuous

In [13]:

# ===== NON-SNN VISUAL ENCODER =====
# Pipeline so s?nh: gi? input/output gi?ng SpikingViTEncoder nh?ng kh?ng d?ng LIF/PLIF/surrogate.
# Input : (B, 1, T, 112, 112)
# Output: (B, T, 512)

class NonSpikingDirectEncoder(nn.Module):
    def __init__(self, in_channels=1, out_channels=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, 45, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3), bias=False),
            nn.BatchNorm3d(45),
            nn.SiLU(inplace=True),
            nn.Conv3d(45, out_channels, kernel_size=(5, 1, 1), stride=(1, 1, 1), padding=(2, 0, 0), bias=False),
            nn.BatchNorm3d(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, X):
        return self.net(X)


class NonSpikingConv2DPlus1D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: int, stride: int=1, padding: int=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=mid_channels,
                kernel_size=(1, 3, 3),
                stride=(1, stride, stride),
                padding=(0, padding, padding),
                bias=False,
            ),
            nn.BatchNorm3d(mid_channels),
            nn.SiLU(inplace=True),
            nn.Conv3d(
                in_channels=mid_channels,
                out_channels=out_channels,
                kernel_size=(3, 1, 1),
                stride=(1, 1, 1),
                padding=(1, 0, 0),
                bias=False,
            ),
        )

    def forward(self, X):
        return self.net(X)

    @staticmethod
    def get_stride(stride: int):
        return stride, stride, stride


class NonSpikingBasicBlock(nn.Module):
    expansion: int = 1

    def __init__(self, in_channels: int, out_channels: int, conv_builder: Callable[..., nn.Module],
                 stride: int = 1, downsample: Optional[nn.Module] = None):
        super().__init__()
        mid_channels = (in_channels * out_channels * 3 * 3 * 3) // (
            in_channels * out_channels * 1 * 3 * 3 + in_channels * out_channels * 3 * 1 * 1
        )
        self.conv1 = nn.Sequential(
            conv_builder(in_channels, out_channels, mid_channels, stride),
            nn.BatchNorm3d(out_channels),
            nn.SiLU(inplace=True),
        )
        self.conv2 = nn.Sequential(
            conv_builder(out_channels, out_channels, mid_channels),
            nn.BatchNorm3d(out_channels),
        )
        self.activation = nn.SiLU(inplace=True)
        self.downsample = downsample

    def forward(self, X: Tensor) -> Tensor:
        residual = X
        out = self.conv1(X)
        out = self.conv2(out)
        if self.downsample is not None:
            residual = self.downsample(X)
        out = out + residual
        return self.activation(out)


class NonSpikingVidResNet(nn.Module):
    def __init__(self, block=NonSpikingBasicBlock, conv_makers=[NonSpikingConv2DPlus1D] * 4,
                 layers=[2, 2, 2, 2], zero_init_residual: bool=False):
        super().__init__()
        self.in_channels = 64
        self.stem = NonSpikingDirectEncoder(in_channels=1, out_channels=64)
        self.layer1 = self._make_layer(block, 64, conv_makers[0], layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, conv_makers[1], layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, conv_makers[2], layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, conv_makers[3], layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool3d((None, 1, 1))
        self.fc = nn.Linear(512 * block.expansion, 512)

        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, NonSpikingBasicBlock):
                    nn.init.constant_(m.conv2[-1].weight, 0)

    def _make_layer(self, block, out_channels: int, conv_builder: Callable[..., nn.Module],
                    num_blocks: int, stride: int) -> nn.Sequential:
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv3d(self.in_channels, out_channels * block.expansion,
                          kernel_size=1, stride=(1, stride, stride), bias=False),
                nn.BatchNorm3d(out_channels * block.expansion),
            )
        layers = [block(self.in_channels, out_channels, conv_builder, stride, downsample)]
        self.in_channels = out_channels * block.expansion
        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels, conv_builder))
        return nn.Sequential(*layers)

    def forward(self, X: Tensor) -> Tensor:
        X = self.stem(X)
        X = self.layer1(X)
        X = self.layer2(X)
        X = self.layer3(X)
        X = self.layer4(X)
        X = self.avgpool(X)
        X = X.squeeze(-1).squeeze(-1)  # (B, C, T)
        X = X.permute(0, 2, 1)        # (B, T, C)
        return self.fc(X)


class NonSpikingTemporalEncoder(nn.Module):
    def __init__(self, z_dim=512, n_heads=8, n_blocks=4, mlp_hidden_dim=2048, dropout=0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=z_dim,
            nhead=n_heads,
            dim_feedforward=mlp_hidden_dim,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_blocks)

    def forward(self, X):
        return self.encoder(X)


class NonSpikingViTEncoder(nn.Module):
    def __init__(self, z_dim=512, n_heads=8, n_blocks=4, mlp_hidden_dim=2048, T_max=1000, dropout=0.1):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, T_max, z_dim) * 0.02)
        self.backbone = NonSpikingVidResNet()
        self.temporal = NonSpikingTemporalEncoder(
            z_dim=z_dim,
            n_heads=n_heads,
            n_blocks=n_blocks,
            mlp_hidden_dim=mlp_hidden_dim,
            dropout=dropout,
        )
        self.norm = nn.LayerNorm(z_dim)

    def forward(self, X):
        features = self.backbone(X)  # (B, T, 512)
        if features.size(1) > self.pos_embedding.size(1):
            raise ValueError(f"T={features.size(1)} v??t T_max={self.pos_embedding.size(1)}")
        features = features + self.pos_embedding[:, :features.size(1), :]
        features = self.temporal(features)
        return self.norm(features)


def build_encoder(encoder_type="snn", **kwargs):
    encoder_type = encoder_type.lower()
    if encoder_type == "snn":
        return SpikingViTEncoder(**kwargs)
    if encoder_type in {"non_snn", "nonsnn", "cnn_transformer"}:
        return NonSpikingViTEncoder(**kwargs)
    raise ValueError(f"Unknown ENCODER_TYPE: {encoder_type}")


# STEP 4: CONTINUOUS INR DECODER

## FiLM Layer

In [14]:
class FiLMLayer(nn.Module):
    def __init__(self, z_dim=512, condition_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(condition_dim, condition_dim),
            nn.ReLU(),
            nn.Linear(condition_dim, 2 * z_dim)
        )
        nn.init.kaiming_uniform_(self.net[-1].weight)
        self.net[-1].weight.data *= 0.01
        self.net[-1].bias.data.uniform_(-1.0 / z_dim, 1.0 / z_dim)
    
    def forward(self, X, Condition):
        gamma, beta = self.net(Condition).chunk(2, dim=-1)

        shape = [1] * (X.ndim - 2) 
        gamma = gamma.view(gamma.size(0), -1, *shape)
        beta = beta.view(beta.size(0), -1, *shape)
        return X * gamma + beta

## From FiLM Layer to TFiLM Layer

In [15]:
class TFiLM(nn.Module):
    def __init__(self, z_dim=512, condition_dim=512):
        super().__init__()
        self.film = FiLMLayer(z_dim=z_dim, condition_dim=condition_dim)

    def forward(self, X, Condition):
        B, T = X.shape[:2]
        X_flat = X.reshape(B * T, *X.shape[2:])
        Condition_flat = Condition.reshape(B * T, *Condition.shape[2:])
        out_flat = self.film(X_flat, Condition_flat)
        out = out_flat.reshape(B, T, *X.shape[2:])
        return out

## SIREN Network

In [16]:
class FiLMSIREN(nn.Module):
    def __init__(self, in_features, out_features, omega_zero=30.0, is_first=False):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.omega_zero = omega_zero
        self.is_first = is_first
        with torch.no_grad():
            if self.is_first:
                bound = 1 / in_features
            else:
                bound = torch.sqrt(torch.tensor(6.0 / in_features)) / self.omega_zero
            
            self.linear.weight.uniform_(-bound, bound)

    def forward(self, X, gamma, beta):
        X = self.linear(X)
        if not self.is_first:
            X = self.omega_zero * X
        out = gamma * X + beta
        return torch.sin(out)

## TFiLM + SIREN -> DECODER

In [17]:
class TFiLMSIRENDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=640, num_layers=4, omega_zero=30.0, use_conv=False, output_activation="tanh"):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.output_activation = output_activation

        total_params = num_layers * 2 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.kaiming_uniform_(self.param_net.weight)
            self.param_net.weight.data *= 0.01
            with torch.no_grad():
                self.param_net.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
        else:
            self.param_gru = nn.GRU(
                condition_dim, condition_dim,
                batch_first=True, bidirectional=True
            )
            self.param_proj = nn.Linear(condition_dim * 2, total_params)
            # Init: Kaiming * 0.01 theo SIREN paper
            nn.init.kaiming_uniform_(self.param_proj.weight)
            self.param_proj.weight.data *= 0.01
            self.param_proj.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
            #     nn.Linear(condition_dim, condition_dim),
            #     nn.ReLU(),
            #     nn.Linear(condition_dim, total_params)
            # )

        self.siren_layers = nn.ModuleList()
        self.siren_layers.append(FiLMSIREN(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=True))
        for _ in range(1, num_layers):
            self.siren_layers.append(FiLMSIREN(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=False))
        self.final_layer = nn.Linear(hidden_dim , out_dim)
        self.output_scale = nn.Parameter(torch.ones(1))  # learnable scale
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)
        self.condition_proj = nn.Linear(condition_dim, hidden_dim)

        # Time positional encoding. This is important for mel output; otherwise the
        # decoder tends to learn only an average spectral envelope.
        self.time_embed = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
    
    def forward(self, Condition):
        B, T, _ = Condition.shape
        if hasattr(self, 'param_net') and isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            gru_out, _ = self.param_gru(Condition)  # (B, T, condition_dim*2)
            params = self.param_proj(gru_out.reshape(B * T, -1))  # (B*T, total_params)
            params = params.reshape(B, T, -1)
        
        gammas = []
        betas = []

        for i in range(self.num_layers):
            start = i * 2 * self.hidden_dim
            beta = params[:, :, start: start + self.hidden_dim]
            gamma = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]
            gammas.append(gamma)
            betas.append(beta)
        
        # Time positional encoding: mỗi frame nhận input khác nhau
        X_base = self.input_constant.expand(B * T, -1)
        # Tạo sinusoidal time positions
        t_pos = torch.linspace(0, 1, T, device=Condition.device)  # [0, 1]
        t_pos = t_pos.unsqueeze(0).expand(B, -1).reshape(B * T, 1)  # (B*T, 1)
        # Sinusoidal encoding: [sin(2^0 * pi * t), cos(2^0 * pi * t), ...]
        # Bounded sinusoidal frequencies. The previous 2**arange(hidden_dim//2)
        # overflowed for hidden_dim=256 and produced sin(inf)=nan.
        half_dim = self.hidden_dim // 2
        pe_dtype = torch.float32
        freqs = torch.linspace(1.0, 32.0, half_dim, device=Condition.device, dtype=pe_dtype)
        angles = t_pos.to(dtype=pe_dtype) * freqs.unsqueeze(0) * torch.pi
        pe = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # (B*T, hidden_dim)
        if pe.shape[-1] < self.hidden_dim:
            pe = torch.nn.functional.pad(pe, (0, self.hidden_dim - pe.shape[-1]))
        pe = pe.to(dtype=Condition.dtype)
        time_signal = self.time_embed(pe)  # (B*T, hidden_dim)
        X = X_base + time_signal  # Mỗi frame có input khác nhau
        gammas_flat = [g.reshape(B * T, -1) for g in gammas]
        betas_flat = [b.reshape(B * T, -1) for b in betas]

        for i, layer in enumerate(self.siren_layers):
            X = layer(X, gammas_flat[i], betas_flat[i])
        out = self.final_layer(X)
        if self.output_activation == "tanh":
            out = torch.tanh(out) * self.output_scale  # waveform range
        elif self.output_activation not in (None, "none", "linear"):
            raise ValueError(f"Unsupported output_activation={self.output_activation}")
        out = out.reshape(B, T, -1)
        return out


# New approach: TFiLM WIRE DECODER

## FiLM + WIRE

In [18]:
class FiLMWIRE(nn.Module):
    def __init__(self, in_features, out_features, omega_zero=30.0, scale=1.0, is_first=False):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.omega_zero = omega_zero
        self.scale = scale
        self.is_first = is_first
        with torch.no_grad():
            if self.is_first:
                bound = 1 / in_features
            else:
                bound = torch.sqrt(torch.tensor(6.0 / in_features)) / self.omega_zero
            self.linear.weight.uniform_(-bound, bound)

    def forward(self, X, gamma, beta):
        X = self.linear(X)
        if not self.is_first:
            X = self.omega_zero * X 
        modulated = gamma * X + beta
        real = torch.cos(self.omega_zero * modulated) * torch.exp(-modulated**2 / (2 * self.scale**2))
        return real


## FiLM WIRE DECODER

In [19]:
class TFiLMWIREDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=640, num_layers=4, omega_zero=30.0, scale=5.0, use_conv=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        total_params = num_layers * 2 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.kaiming_uniform_(self.param_net.weight)
            self.param_net.weight.data *= 0.01
            with torch.no_grad():
                self.param_net.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
        else:
            self.param_gru = nn.GRU(
                condition_dim, condition_dim,
                batch_first=True, bidirectional=True
            )
            self.param_proj = nn.Linear(condition_dim * 2, total_params)
            # Init: Kaiming * 0.01 theo SIREN paper
            nn.init.kaiming_uniform_(self.param_proj.weight)
            self.param_proj.weight.data *= 0.01
            self.param_proj.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
            #     nn.Linear(condition_dim, condition_dim),
            #     nn.ReLU(),
            #     nn.Linear(condition_dim, total_params)
            # )

        self.wire_layers = nn.ModuleList()
        self.wire_layers.append(FiLMWIRE(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, scale=scale, is_first=True))
        for _ in range(1, num_layers):
            self.wire_layers.append(FiLMWIRE(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, scale=scale, is_first=False))
        self.final_layer = nn.Linear(hidden_dim , out_dim)
        self.output_scale = nn.Parameter(torch.ones(1))  # learnable scale
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)

        # Time positional encoding
        self.time_embed = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
    
    def forward(self, Condition):
        B, T, _ = Condition.shape
        if hasattr(self, 'param_net') and isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            gru_out, _ = self.param_gru(Condition)  # (B, T, condition_dim*2)
            params = self.param_proj(gru_out.reshape(B * T, -1))  # (B*T, total_params)
            params = params.reshape(B, T, -1)
        
        gammas = []
        betas = []

        for i in range(self.num_layers):
            start = i * 2 * self.hidden_dim
            beta = params[:, :, start: start + self.hidden_dim]
            gamma = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]
            gammas.append(gamma)
            betas.append(beta)
        
        # Time positional encoding: mỗi frame nhận input khác nhau
        X_base = self.input_constant.expand(B * T, -1)
        # Tạo sinusoidal time positions
        t_pos = torch.linspace(0, 1, T, device=Condition.device)  # [0, 1]
        t_pos = t_pos.unsqueeze(0).expand(B, -1).reshape(B * T, 1)  # (B*T, 1)
        # Sinusoidal encoding: [sin(2^0 * pi * t), cos(2^0 * pi * t), ...]
        # Bounded sinusoidal frequencies. The previous 2**arange(hidden_dim//2)
        # overflowed for hidden_dim=256 and produced sin(inf)=nan.
        half_dim = self.hidden_dim // 2
        pe_dtype = torch.float32
        freqs = torch.linspace(1.0, 32.0, half_dim, device=Condition.device, dtype=pe_dtype)
        angles = t_pos.to(dtype=pe_dtype) * freqs.unsqueeze(0) * torch.pi
        pe = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # (B*T, hidden_dim)
        if pe.shape[-1] < self.hidden_dim:
            pe = torch.nn.functional.pad(pe, (0, self.hidden_dim - pe.shape[-1]))
        pe = pe.to(dtype=Condition.dtype)
        time_signal = self.time_embed(pe)  # (B*T, hidden_dim)
        X = X_base + time_signal  # Mỗi frame có input khác nhau
        gammas_flat = [g.reshape(B * T, -1) for g in gammas]
        betas_flat = [b.reshape(B * T, -1) for b in betas]

        for i, layer in enumerate(self.wire_layers):
            X = layer(X, gammas_flat[i], betas_flat[i])
        out = self.final_layer(X)
        out = torch.tanh(out) * self.output_scale  # [-scale, +scale], waveform range
        out = out.reshape(B, T, -1)
        return out


# New approach: WIRE + SIREN DUAL DECODER

## Dual Layer

In [20]:
class DualFiLMLayer(nn.Module):
    def __init__(self, in_features, out_features, omega_siren=30.0, omega_wire=30.0, scale=5.0, is_first=False):
        super().__init__()
        self.siren = FiLMSIREN(in_features, out_features, omega_siren, is_first)
        self.wire = FiLMWIRE(in_features, out_features, omega_wire, scale, is_first)
        self.fusion = nn.Sequential(
            nn.Linear(out_features * 2, out_features),
            nn.ReLU(),
            nn.Linear(out_features, out_features)
        )

    def forward(self, X, gamma_s, beta_s, gamma_w, beta_w):
        out_siren = self.siren(X, gamma_s, beta_s)
        out_wire = self.wire(X, gamma_w, beta_w)
        out = torch.cat([out_siren, out_wire], dim=-1)
        out = self.fusion(out)
        return out


## Dual Decoder

In [21]:
class DualDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=640, num_layers=4, omega_siren=30.0, omega_wire=30.0, scale=5.0, use_conv=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        total_params = self.num_layers * 4 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.kaiming_uniform_(self.param_net.weight)
            self.param_net.weight.data *= 0.01
            with torch.no_grad():
                self.param_net.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)

        else: 
            self.param_gru = nn.GRU(
                condition_dim, condition_dim,
                batch_first=True, bidirectional=True
            )
            self.param_proj = nn.Linear(condition_dim * 2, total_params)
            # Init: Kaiming * 0.01 theo SIREN paper
            nn.init.kaiming_uniform_(self.param_proj.weight)
            self.param_proj.weight.data *= 0.01
            self.param_proj.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
            #     nn.Linear(condition_dim, condition_dim),
            #     nn.ReLU(),
            #     nn.Linear(condition_dim, total_params)
            # )

        self.dual_layers = nn.ModuleList()
        self.dual_layers.append(DualFiLMLayer(hidden_dim, hidden_dim, omega_siren, omega_wire, scale, is_first=True))
        for _ in range(1, num_layers):
            self.dual_layers.append(DualFiLMLayer(hidden_dim, hidden_dim, omega_siren, omega_wire, scale, is_first=False))
        self.final_layer = nn.Linear(hidden_dim, out_dim)
        self.output_scale = nn.Parameter(torch.ones(1))  # learnable scale
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)

        # Time positional encoding
        self.time_embed = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, Condition):
        B, T, _ = Condition.shape
        if hasattr(self, 'param_net') and isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            gru_out, _ = self.param_gru(Condition)  # (B, T, condition_dim*2)
            params = self.param_proj(gru_out.reshape(B * T, -1))  # (B*T, total_params)
            params = params.reshape(B, T, -1)

        gammas_s = []
        betas_s = []
        gammas_w = []
        betas_w = []

        for i in range(self.num_layers):
            start = i * 4 * self.hidden_dim
            # SIREN
            beta_s = params[:, :, start: start + self.hidden_dim]
            gamma_s = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]

            # WIRE
            beta_w = params[:, :, start + 2 * self.hidden_dim: start + 3 * self.hidden_dim]
            gamma_w = params[:, :, start + 3 * self.hidden_dim: start + 4 * self.hidden_dim]

            gammas_s.append(gamma_s)
            betas_s.append(beta_s)
            gammas_w.append(gamma_w)
            betas_w.append(beta_w)

        # Time positional encoding: mỗi frame nhận input khác nhau
        X_base = self.input_constant.expand(B * T, -1)
        # Tạo sinusoidal time positions
        t_pos = torch.linspace(0, 1, T, device=Condition.device)  # [0, 1]
        t_pos = t_pos.unsqueeze(0).expand(B, -1).reshape(B * T, 1)  # (B*T, 1)
        # Sinusoidal encoding: [sin(2^0 * pi * t), cos(2^0 * pi * t), ...]
        # Bounded sinusoidal frequencies. The previous 2**arange(hidden_dim//2)
        # overflowed for hidden_dim=256 and produced sin(inf)=nan.
        half_dim = self.hidden_dim // 2
        pe_dtype = torch.float32
        freqs = torch.linspace(1.0, 32.0, half_dim, device=Condition.device, dtype=pe_dtype)
        angles = t_pos.to(dtype=pe_dtype) * freqs.unsqueeze(0) * torch.pi
        pe = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # (B*T, hidden_dim)
        if pe.shape[-1] < self.hidden_dim:
            pe = torch.nn.functional.pad(pe, (0, self.hidden_dim - pe.shape[-1]))
        pe = pe.to(dtype=Condition.dtype)
        time_signal = self.time_embed(pe)  # (B*T, hidden_dim)
        X = X_base + time_signal  # Mỗi frame có input khác nhau
        
        gammas_s_flat = [g.reshape(B * T, -1) for g in gammas_s]
        betas_s_flat = [b.reshape(B * T, -1) for b in betas_s]
        gammas_w_flat = [g.reshape(B * T, -1) for g in gammas_w]
        betas_w_flat = [b.reshape(B * T, -1) for b in betas_w]

        for i, layer in enumerate(self.dual_layers):
            X = layer(
                X,
                gammas_s_flat[i], betas_s_flat[i],
                gammas_w_flat[i], betas_w_flat[i]
            )
        out = self.final_layer(X)
        out = torch.tanh(out) * self.output_scale  # [-scale, +scale], waveform range
        out = out.reshape(B, T, -1)
        return out
            


# New approach: TFiLM FINER DECODER

## TFiLM FINER Layer

In [22]:
class FiLMFINER(nn.Module):
    def __init__(self, in_features, out_features, omega_zero=30.0, is_first=False):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.omega_zero = omega_zero
        self.is_first = is_first
        with torch.no_grad():
            if self.is_first:
                bound = 1 / in_features
            else:
                bound = torch.sqrt(torch.tensor(6.0 / in_features)) / self.omega_zero
            self.linear.weight.uniform_(-bound, bound)
        
    def forward(self, X, gamma, beta):
        out_dtype = X.dtype
        X = self.linear(X)
        if not self.is_first:
            X = self.omega_zero * X
        modulated = gamma.float() * X.float() + beta.float()
        phase = self.omega_zero * (torch.abs(modulated) + 1.0) * modulated
        phase = torch.nan_to_num(phase, nan=0.0, posinf=1000.0, neginf=-1000.0).clamp(-1000.0, 1000.0)
        return torch.sin(phase).to(dtype=out_dtype)


## TFiLM FINER DECODER

In [23]:
class TFiLMFINERDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=640, num_layers=4, omega_zero=30.0, use_conv=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        total_params = num_layers * 2 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.kaiming_uniform_(self.param_net.weight)
            self.param_net.weight.data *= 0.01
            with torch.no_grad():
                self.param_net.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
        else:
            self.param_gru = nn.GRU(
                condition_dim, condition_dim,
                batch_first=True, bidirectional=True
            )
            self.param_proj = nn.Linear(condition_dim * 2, total_params)
            # Init: Kaiming * 0.01 theo SIREN paper
            nn.init.kaiming_uniform_(self.param_proj.weight)
            self.param_proj.weight.data *= 0.01
            self.param_proj.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
            #     nn.Linear(condition_dim, condition_dim),
            #     nn.ReLU(),
            #     nn.Linear(condition_dim, total_params)
            # )

        self.finer_layers = nn.ModuleList()
        self.finer_layers.append(FiLMFINER(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=True))
        for _ in range(1, num_layers):
            self.finer_layers.append(FiLMFINER(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=False))
        self.final_layer = nn.Linear(hidden_dim , out_dim)
        self.output_scale = nn.Parameter(torch.ones(1))  # learnable scale
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)

        # Time positional encoding
        self.time_embed = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
    
    def forward(self, Condition):
        B, T, _ = Condition.shape
        if hasattr(self, 'param_net') and isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            gru_out, _ = self.param_gru(Condition)  # (B, T, condition_dim*2)
            params = self.param_proj(gru_out.reshape(B * T, -1))  # (B*T, total_params)
            params = params.reshape(B, T, -1)
        
        gammas = []
        betas = []

        for i in range(self.num_layers):
            start = i * 2 * self.hidden_dim
            beta = params[:, :, start: start + self.hidden_dim]
            gamma = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]
            gammas.append(gamma)
            betas.append(beta)
        
        # Time positional encoding: mỗi frame nhận input khác nhau
        X_base = self.input_constant.expand(B * T, -1)
        # Tạo sinusoidal time positions
        t_pos = torch.linspace(0, 1, T, device=Condition.device)  # [0, 1]
        t_pos = t_pos.unsqueeze(0).expand(B, -1).reshape(B * T, 1)  # (B*T, 1)
        # Sinusoidal encoding: [sin(2^0 * pi * t), cos(2^0 * pi * t), ...]
        # Bounded sinusoidal frequencies. The previous 2**arange(hidden_dim//2)
        # overflowed for hidden_dim=256 and produced sin(inf)=nan.
        half_dim = self.hidden_dim // 2
        pe_dtype = torch.float32
        freqs = torch.linspace(1.0, 32.0, half_dim, device=Condition.device, dtype=pe_dtype)
        angles = t_pos.to(dtype=pe_dtype) * freqs.unsqueeze(0) * torch.pi
        pe = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # (B*T, hidden_dim)
        if pe.shape[-1] < self.hidden_dim:
            pe = torch.nn.functional.pad(pe, (0, self.hidden_dim - pe.shape[-1]))
        pe = pe.to(dtype=Condition.dtype)
        time_signal = self.time_embed(pe)  # (B*T, hidden_dim)
        X = X_base + time_signal  # Mỗi frame có input khác nhau
        gammas_flat = [g.reshape(B * T, -1) for g in gammas]
        betas_flat = [b.reshape(B * T, -1) for b in betas]

        for i, layer in enumerate(self.finer_layers):
            X = layer(X, gammas_flat[i], betas_flat[i])
        out = self.final_layer(X)
        out = torch.tanh(out) * self.output_scale  # [-scale, +scale], waveform range
        out = out.reshape(B, T, -1)
        return out


# New approach: Wrap FINER SIREN DECODER

## FINER SIREN WRAP

In [24]:
# out = sin((∣b~∣+1)⋅(ω0(Wx+b)))
class FiLMWrapFINSIREN(nn.Module):
    def __init__(self, in_features, out_features, omega_zero=30.0, is_first=False):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=True)
        self.omega_zero = omega_zero
        self.is_first = is_first
        self.freq_bias = nn.Parameter(torch.empty(out_features))
        self.reset_parameters()
    
    def reset_parameters(self):
        with torch.no_grad():
            if self.is_first:
                bound = 1.0 / self.linear.in_features
            else:
                bound = torch.sqrt(torch.tensor(6.0 / self.linear.in_features)) / self.omega_zero
            self.linear.weight.uniform_(-bound, bound)
            nn.init.zeros_(self.linear.bias)
            nn.init.uniform_(self.freq_bias, -10.0, 10.0)
    
    def forward(self, X, gamma, beta):
        X = self.linear(X)
        if not self.is_first:
            X = self.omega_zero * X
        modulated = gamma * X + beta
        scaled = torch.abs(self.freq_bias) + 1.0
        out = torch.sin(scaled.unsqueeze(0) * modulated)
        return out

## TFiLM WRAP FIN SIN

In [25]:
class TFiLMWrapFISINDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=640, num_layers=4, omega_zero=30.0, use_conv=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        total_params = num_layers * 2 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.kaiming_uniform_(self.param_net.weight)
            self.param_net.weight.data *= 0.01
            with torch.no_grad():
                self.param_net.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
        else:
            self.param_gru = nn.GRU(
                condition_dim, condition_dim,
                batch_first=True, bidirectional=True
            )
            self.param_proj = nn.Linear(condition_dim * 2, total_params)
            # Init: Kaiming * 0.01 theo SIREN paper
            nn.init.kaiming_uniform_(self.param_proj.weight)
            self.param_proj.weight.data *= 0.01
            self.param_proj.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
            #     nn.Linear(condition_dim, condition_dim),
            #     nn.ReLU(),
            #     nn.Linear(condition_dim, total_params)
            # )

        self.wrap_layers = nn.ModuleList()
        self.wrap_layers.append(FiLMWrapFINSIREN(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=True))
        for _ in range(1, num_layers):
            self.wrap_layers.append(FiLMWrapFINSIREN(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=False))
        self.final_layer = nn.Linear(hidden_dim , out_dim)
        self.output_scale = nn.Parameter(torch.ones(1))  # learnable scale
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)

        # Time positional encoding
        self.time_embed = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
    
    def forward(self, Condition):
        B, T, _ = Condition.shape
        if hasattr(self, 'param_net') and isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            gru_out, _ = self.param_gru(Condition)  # (B, T, condition_dim*2)
            params = self.param_proj(gru_out.reshape(B * T, -1))  # (B*T, total_params)
            params = params.reshape(B, T, -1)
        
        gammas = []
        betas = []

        for i in range(self.num_layers):
            start = i * 2 * self.hidden_dim
            beta = params[:, :, start: start + self.hidden_dim]
            gamma = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]
            gammas.append(gamma)
            betas.append(beta)
        
        # Time positional encoding: mỗi frame nhận input khác nhau
        X_base = self.input_constant.expand(B * T, -1)
        # Tạo sinusoidal time positions
        t_pos = torch.linspace(0, 1, T, device=Condition.device)  # [0, 1]
        t_pos = t_pos.unsqueeze(0).expand(B, -1).reshape(B * T, 1)  # (B*T, 1)
        # Sinusoidal encoding: [sin(2^0 * pi * t), cos(2^0 * pi * t), ...]
        # Bounded sinusoidal frequencies. The previous 2**arange(hidden_dim//2)
        # overflowed for hidden_dim=256 and produced sin(inf)=nan.
        half_dim = self.hidden_dim // 2
        pe_dtype = torch.float32
        freqs = torch.linspace(1.0, 32.0, half_dim, device=Condition.device, dtype=pe_dtype)
        angles = t_pos.to(dtype=pe_dtype) * freqs.unsqueeze(0) * torch.pi
        pe = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # (B*T, hidden_dim)
        if pe.shape[-1] < self.hidden_dim:
            pe = torch.nn.functional.pad(pe, (0, self.hidden_dim - pe.shape[-1]))
        pe = pe.to(dtype=Condition.dtype)
        time_signal = self.time_embed(pe)  # (B*T, hidden_dim)
        X = X_base + time_signal  # Mỗi frame có input khác nhau
        gammas_flat = [g.reshape(B * T, -1) for g in gammas]
        betas_flat = [b.reshape(B * T, -1) for b in betas]

        for i, layer in enumerate(self.wrap_layers):
            X = layer(X, gammas_flat[i], betas_flat[i])
        out = self.final_layer(X)
        out = torch.tanh(out) * self.output_scale  # [-scale, +scale], waveform range
        out = out.reshape(B, T, -1)
        return out


# New approach: Wrap FINER WIRE DECODER

## FINER WIRE WRAP

In [26]:
class FiLMWrapFINWIRE(nn.Module):
    def __init__(self, in_features, out_features, omega_zero=30.0, scale=5.0, is_first=False):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.omega_zero = omega_zero
        self.scale = scale
        self.is_first = is_first
        self.freq_bias = nn.Parameter(torch.empty(out_features))
        with torch.no_grad():
            if self.is_first:
                bound = 1 / self.linear.in_features
            else:
                bound = torch.sqrt(torch.tensor(6.0 / in_features)) / self.omega_zero
            self.linear.weight.uniform_(-bound, bound)
            nn.init.zeros_(self.linear.bias)
            nn.init.uniform_(self.freq_bias, -10, 10)

    def forward(self, X, gamma, beta):
        X = self.linear(X)
        if not self.is_first:
            X = self.omega_zero * X 
        modulated = gamma * X + beta
        scaled = torch.abs(self.freq_bias) + 1.0
        real = torch.cos(scaled.unsqueeze(0) * modulated) * torch.exp(-modulated**2 / (2 * self.scale**2))
        return real


## TFiLM WRAP FIN WI DEOCDER

In [27]:
class TFiLMWrapFIWIDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=640, num_layers=4, omega_zero=30.0, scale=5.0, use_conv=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        total_params = num_layers * 2 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.kaiming_uniform_(self.param_net.weight)
            self.param_net.weight.data *= 0.01
            with torch.no_grad():
                self.param_net.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
        else:
            self.param_gru = nn.GRU(
                condition_dim, condition_dim,
                batch_first=True, bidirectional=True
            )
            self.param_proj = nn.Linear(condition_dim * 2, total_params)
            # Init: Kaiming * 0.01 theo SIREN paper
            nn.init.kaiming_uniform_(self.param_proj.weight)
            self.param_proj.weight.data *= 0.01
            self.param_proj.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
            #     nn.Linear(condition_dim, condition_dim),
            #     nn.ReLU(),
            #     nn.Linear(condition_dim, total_params)
            # )

        self.wrap_layers = nn.ModuleList()
        self.wrap_layers.append(FiLMWrapFINWIRE(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, scale=scale, is_first=True))
        for _ in range(1, num_layers):
            self.wrap_layers.append(FiLMWrapFINWIRE(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, scale=scale, is_first=False))
        self.final_layer = nn.Linear(hidden_dim , out_dim)
        self.output_scale = nn.Parameter(torch.ones(1))  # learnable scale
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)

        # Time positional encoding
        self.time_embed = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
    
    def forward(self, Condition):
        B, T, _ = Condition.shape
        if hasattr(self, 'param_net') and isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            gru_out, _ = self.param_gru(Condition)  # (B, T, condition_dim*2)
            params = self.param_proj(gru_out.reshape(B * T, -1))  # (B*T, total_params)
            params = params.reshape(B, T, -1)
        
        gammas = []
        betas = []

        for i in range(self.num_layers):
            start = i * 2 * self.hidden_dim
            beta = params[:, :, start: start + self.hidden_dim]
            gamma = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]
            gammas.append(gamma)
            betas.append(beta)
        
        # Time positional encoding: mỗi frame nhận input khác nhau
        X_base = self.input_constant.expand(B * T, -1)
        # Tạo sinusoidal time positions
        t_pos = torch.linspace(0, 1, T, device=Condition.device)  # [0, 1]
        t_pos = t_pos.unsqueeze(0).expand(B, -1).reshape(B * T, 1)  # (B*T, 1)
        # Sinusoidal encoding: [sin(2^0 * pi * t), cos(2^0 * pi * t), ...]
        # Bounded sinusoidal frequencies. The previous 2**arange(hidden_dim//2)
        # overflowed for hidden_dim=256 and produced sin(inf)=nan.
        half_dim = self.hidden_dim // 2
        pe_dtype = torch.float32
        freqs = torch.linspace(1.0, 32.0, half_dim, device=Condition.device, dtype=pe_dtype)
        angles = t_pos.to(dtype=pe_dtype) * freqs.unsqueeze(0) * torch.pi
        pe = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # (B*T, hidden_dim)
        if pe.shape[-1] < self.hidden_dim:
            pe = torch.nn.functional.pad(pe, (0, self.hidden_dim - pe.shape[-1]))
        pe = pe.to(dtype=Condition.dtype)
        time_signal = self.time_embed(pe)  # (B*T, hidden_dim)
        X = X_base + time_signal  # Mỗi frame có input khác nhau
        gammas_flat = [g.reshape(B * T, -1) for g in gammas]
        betas_flat = [b.reshape(B * T, -1) for b in betas]

        for i, layer in enumerate(self.wrap_layers):
            X = layer(X, gammas_flat[i], betas_flat[i])
        out = self.final_layer(X)
        out = torch.tanh(out) * self.output_scale  # [-scale, +scale], waveform range
        out = out.reshape(B, T, -1)
        return out


# New approach: Dual Wrap FINER - SIREN WIRE

## Dual Wrap Layer

In [28]:
class DualWrapLayer(nn.Module):
    def __init__(self, in_features, out_features, omega_fisin=30.0, omega_fiwi=30.0, scale=5.0, is_first=False):
        super().__init__()
        self.fisin = FiLMWrapFINSIREN(in_features, out_features, omega_fisin, is_first)
        self.fiwi = FiLMWrapFINWIRE(in_features, out_features, omega_fiwi, scale, is_first)
        self.fusion = nn.Sequential(
            nn.Linear(out_features * 2, out_features),
            nn.ReLU(),
            nn.Linear(out_features, out_features)
        )
    
    def forward(self, X, gamma_fs, beta_fs, gamma_fw, beta_fw):
        out_fs = self.fisin(X, gamma_fs, beta_fs)
        out_fw = self.fiwi(X, gamma_fw, beta_fw)
        out = torch.cat([out_fs, out_fw], dim=-1)
        out = self.fusion(out)
        return out

## Dual Wrap Decoder

In [29]:
class DualWrapDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=640, num_layers=4, omega_siren=30.0, omega_wire=30.0, scale=5.0, use_conv=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        total_params = self.num_layers * 4 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.kaiming_uniform_(self.param_net.weight)
            self.param_net.weight.data *= 0.01
            with torch.no_grad():
                self.param_net.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)

        else: 
            self.param_gru = nn.GRU(
                condition_dim, condition_dim,
                batch_first=True, bidirectional=True
            )
            self.param_proj = nn.Linear(condition_dim * 2, total_params)
            # Init: Kaiming * 0.01 theo SIREN paper
            nn.init.kaiming_uniform_(self.param_proj.weight)
            self.param_proj.weight.data *= 0.01
            self.param_proj.bias.data.uniform_(-1.0 / hidden_dim, 1.0 / hidden_dim)
            #     nn.Linear(condition_dim, condition_dim),
            #     nn.ReLU(),
            #     nn.Linear(condition_dim, total_params)
            # )

        self.dual_layers = nn.ModuleList()
        self.dual_layers.append(DualWrapLayer(hidden_dim, hidden_dim, omega_siren, omega_wire, scale, is_first=True))
        for _ in range(1, num_layers):
            self.dual_layers.append(DualWrapLayer(hidden_dim, hidden_dim, omega_siren, omega_wire, scale, is_first=False))
        self.final_layer = nn.Linear(hidden_dim, out_dim)
        self.output_scale = nn.Parameter(torch.ones(1))  # learnable scale
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)

        # Time positional encoding
        self.time_embed = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, Condition):
        B, T, _ = Condition.shape
        if hasattr(self, 'param_net') and isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            gru_out, _ = self.param_gru(Condition)  # (B, T, condition_dim*2)
            params = self.param_proj(gru_out.reshape(B * T, -1))  # (B*T, total_params)
            params = params.reshape(B, T, -1)

        gammas_s = []
        betas_s = []
        gammas_w = []
        betas_w = []

        for i in range(self.num_layers):
            start = i * 4 * self.hidden_dim
            # SIREN
            beta_s = params[:, :, start: start + self.hidden_dim]
            gamma_s = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]

            # WIRE
            beta_w = params[:, :, start + 2 * self.hidden_dim: start + 3 * self.hidden_dim]
            gamma_w = params[:, :, start + 3 * self.hidden_dim: start + 4 * self.hidden_dim]

            gammas_s.append(gamma_s)
            betas_s.append(beta_s)
            gammas_w.append(gamma_w)
            betas_w.append(beta_w)

        # Time positional encoding: mỗi frame nhận input khác nhau
        X_base = self.input_constant.expand(B * T, -1)
        # Tạo sinusoidal time positions
        t_pos = torch.linspace(0, 1, T, device=Condition.device)  # [0, 1]
        t_pos = t_pos.unsqueeze(0).expand(B, -1).reshape(B * T, 1)  # (B*T, 1)
        # Sinusoidal encoding: [sin(2^0 * pi * t), cos(2^0 * pi * t), ...]
        # Bounded sinusoidal frequencies. The previous 2**arange(hidden_dim//2)
        # overflowed for hidden_dim=256 and produced sin(inf)=nan.
        half_dim = self.hidden_dim // 2
        pe_dtype = torch.float32
        freqs = torch.linspace(1.0, 32.0, half_dim, device=Condition.device, dtype=pe_dtype)
        angles = t_pos.to(dtype=pe_dtype) * freqs.unsqueeze(0) * torch.pi
        pe = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # (B*T, hidden_dim)
        if pe.shape[-1] < self.hidden_dim:
            pe = torch.nn.functional.pad(pe, (0, self.hidden_dim - pe.shape[-1]))
        pe = pe.to(dtype=Condition.dtype)
        time_signal = self.time_embed(pe)  # (B*T, hidden_dim)
        X = X_base + time_signal  # Mỗi frame có input khác nhau
        gammas_s_flat = [g.reshape(B * T, -1) for g in gammas_s]
        betas_s_flat = [b.reshape(B * T, -1) for b in betas_s]
        gammas_w_flat = [g.reshape(B * T, -1) for g in gammas_w]
        betas_w_flat = [b.reshape(B * T, -1) for b in betas_w]

        for i, layer in enumerate(self.dual_layers):
            X = layer(
                X,
                gammas_s_flat[i], betas_s_flat[i],
                gammas_w_flat[i], betas_w_flat[i]
            )
        out = self.final_layer(X)
        out = torch.tanh(out) * self.output_scale  # [-scale, +scale], waveform range
        out = out.reshape(B, T, -1)
        return out
            


# MIGRATE LANDMARKS INTO EXISTING .pt


In [ ]:
# ===== MIGRATE LANDMARKS INTO EXISTING .pt FILES =====
# Use this when your mel .pt files are already correct and you only want to add lip landmarks.
# This cell does NOT recompute mel and does NOT import SpeechBrain.

import os
import glob
import time
import gc
import subprocess
import numpy as np
import torch
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from decord import VideoReader, cpu
from tqdm.notebook import tqdm

MIGRATE_LM_PT_DIR = globals().get('MEL_DATA_DIR', 'Processed_Data_Mel_HiFiGAN')
MIGRATE_LM_DATASET_PATH = 'Dataset_Output'
MIGRATE_LM_IN_PLACE = True  # set True after testing
MIGRATE_LM_LIMIT = None       # set None to run all files
MIGRATE_LM_FORCE = True     # set True to overwrite existing landmarks
MIGRATE_LM_CONVERT_FPS = False  # source clips are already aligned; keep False to avoid slow ffmpeg temp files
MIGRATE_LM_FPS = 25
MIGRATE_LM_MODEL_PATH = 'face_landmarker_v2_with_blendshapes.task'

LIP_LANDMARK_IDXS = [
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291,
    185, 40, 39, 37, 0, 267, 269, 270, 409,
    78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
    191, 80, 81, 82, 13, 312, 311, 310, 415,
]


def safe_name_from_folder(folder_path):
    parts = folder_path.split(os.sep)
    return f"{parts[-2]}_{parts[-1]}" if len(parts) >= 2 else parts[-1]


def build_source_folder_map(dataset_path):
    video_paths = glob.glob(os.path.join(dataset_path, '**', 'video.mp4'), recursive=True)
    folder_map = {}
    for video_path in video_paths:
        folder = os.path.dirname(video_path)
        folder_map[safe_name_from_folder(folder)] = folder
    return folder_map


def convert_video_fps(input_path, output_path, target_fps=25):
    subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-i', input_path,
        '-r', str(target_fps),
        '-vcodec', 'libx264',
        '-pix_fmt', 'yuv420p',
        '-crf', '23',
        '-preset', 'fast',
        output_path,
    ], check=True, capture_output=True)


def safe_remove(path, retries=8, delay=0.25):
    if not path or not os.path.exists(path):
        return True
    for attempt in range(retries):
        try:
            os.remove(path)
            return True
        except PermissionError:
            gc.collect()
            time.sleep(delay * (attempt + 1))
    print(f"  -> Warning: could not remove temp file: {path}")
    return False


def extract_lip_landmarks(lm_list):
    return np.asarray(
        [[lm_list[idx][0], lm_list[idx][1]] for idx in LIP_LANDMARK_IDXS],
        dtype=np.float32,
    )


def resample_landmarks_to_len(landmarks_tensor, target_len):
    # landmarks_tensor: (T, N, 2). Linear interpolate over time if ffmpeg/decord gives a slightly different frame count.
    if landmarks_tensor.shape[0] == target_len:
        return landmarks_tensor
    x = landmarks_tensor.permute(1, 2, 0).reshape(1, -1, landmarks_tensor.shape[0])
    x = torch.nn.functional.interpolate(x, size=int(target_len), mode='linear', align_corners=False)
    return x.reshape(landmarks_tensor.shape[1], landmarks_tensor.shape[2], target_len).permute(2, 0, 1).contiguous()


def extract_landmarks_from_video(video_path, target_video_len=None):
    base_options = python.BaseOptions(model_asset_path=MIGRATE_LM_MODEL_PATH)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1,
    )

    detector = None
    vr = None
    landmark_frames = []
    last_lip_landmarks = None
    try:
        detector = vision.FaceLandmarker.create_from_options(options)
        vr = VideoReader(video_path, ctx=cpu(0))
        for i in range(len(vr)):
            frame = vr[i].asnumpy()
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
            timestamp_ms = int(i * (1000 / MIGRATE_LM_FPS))
            result = detector.detect_for_video(mp_image, timestamp_ms)

            if result.face_landmarks:
                landmarks = result.face_landmarks[0]
                lm_list = [[lm.x, lm.y, lm.z] for lm in landmarks]
                lip_lm = extract_lip_landmarks(lm_list)
                landmark_frames.append(lip_lm)
                last_lip_landmarks = lip_lm
            else:
                if last_lip_landmarks is not None:
                    landmark_frames.append(last_lip_landmarks.copy())
                else:
                    landmark_frames.append(np.zeros((len(LIP_LANDMARK_IDXS), 2), dtype=np.float32))
    finally:
        if detector is not None:
            detector.close()
        if vr is not None:
            del vr
        gc.collect()

    if not landmark_frames:
        raise RuntimeError(f'No frames processed from {video_path}')

    landmarks_tensor = torch.from_numpy(np.stack(landmark_frames)).float()  # (T_video, 40, 2)
    if target_video_len is not None:
        landmarks_tensor = resample_landmarks_to_len(landmarks_tensor, int(target_video_len))
    return landmarks_tensor


folder_map = build_source_folder_map(MIGRATE_LM_DATASET_PATH)
pt_files = sorted([f for f in os.listdir(MIGRATE_LM_PT_DIR) if f.endswith('.pt')])
if MIGRATE_LM_LIMIT is not None:
    pt_files = pt_files[:MIGRATE_LM_LIMIT]

print(f"Found {len(pt_files)} .pt files | in_place={MIGRATE_LM_IN_PLACE} | force={MIGRATE_LM_FORCE} | convert_fps={MIGRATE_LM_CONVERT_FPS}", flush=True)
print(f"Source folders indexed: {len(folder_map)}", flush=True)

for name in tqdm(pt_files):
    pt_path = os.path.join(MIGRATE_LM_PT_DIR, name)
    data = torch.load(pt_path, map_location='cpu', weights_only=False)

    if not MIGRATE_LM_FORCE and ('landmarks' in data or 'lip_landmarks' in data):
        print(f"Skip {name}: already has landmarks", flush=True)
        continue

    safe_name = os.path.splitext(name)[0]
    folder = folder_map.get(safe_name)
    if folder is None:
        print(f"Skip {name}: cannot find source folder for safe_name={safe_name}", flush=True)
        continue

    video_candidates = glob.glob(os.path.join(folder, 'video.*'))
    if not video_candidates:
        print(f"Skip {name}: missing source video in {folder}", flush=True)
        continue

    source_video = video_candidates[0]
    temp_video = os.path.join(folder, 'temp_landmark_25fps.mp4')
    read_video = source_video
    try:
        print(f"Start {name}: source={source_video}", flush=True)
        if MIGRATE_LM_CONVERT_FPS:
            print(f"  converting to {MIGRATE_LM_FPS}fps...", flush=True)
            convert_video_fps(source_video, temp_video, target_fps=MIGRATE_LM_FPS)
            read_video = temp_video
        target_video_len = int(data.get('video_len', data['video'].shape[1]))
        print(f"  extracting landmarks from {read_video} | target_video_len={target_video_len}", flush=True)
        landmarks = extract_landmarks_from_video(read_video, target_video_len=target_video_len)

        data['landmarks'] = landmarks
        data['lip_landmark_idxs'] = torch.tensor(LIP_LANDMARK_IDXS, dtype=torch.long)
        data['landmark_type'] = 'mediapipe_lip_xy_normalized'
        data['landmark_len'] = int(landmarks.shape[0])

        print(f"Done {name}: landmarks={tuple(landmarks.shape)}, video_len={target_video_len}", flush=True)
        if MIGRATE_LM_IN_PLACE:
            torch.save(data, pt_path)
    except Exception as e:
        print(f"ERROR {name}: {e}", flush=True)
        import traceback
        traceback.print_exc()
    finally:
        if MIGRATE_LM_CONVERT_FPS:
            safe_remove(temp_video)

print("Done. If the test looks good, set MIGRATE_LM_IN_PLACE=True and MIGRATE_LM_LIMIT=None, then run again.", flush=True)



## Main process

In [31]:
# ===== MEL TRAINING CONFIG / UTILITIES =====
# Mel is the first experimental target. Landmarks, when enabled, stay aligned to video frames;
# MelTemporalUpsampleDecoder then maps video-frame latents to mel frames.

import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from spikingjelly.activation_based import functional

TARGET_TYPE = "mel_hifigan"  # "mel_hifigan" first; switch to "waveform" later if mel is stable.
MEL_DATA_DIR = "Processed_Data_Mel_HiFiGAN"
WAVEFORM_DATA_DIR = "Processed_Data_Mel_HiFiGAN"
LANDMARK_KEYS = ("lip_landmarks", "mouth_landmarks", "landmarks", "face_landmarks")



def pick_pt_file(data_dir, pt_path=None):
    if pt_path is not None:
        return pt_path
    files = sorted([f for f in os.listdir(data_dir) if f.endswith(".pt")])
    if not files:
        raise FileNotFoundError(f"No .pt files found in {data_dir}")
    return os.path.join(data_dir, files[0])


def find_landmarks_in_data(data, require=False, path=None):
    for key in LANDMARK_KEYS:
        if key in data:
            lm = data[key]
            if not torch.is_tensor(lm):
                lm = torch.as_tensor(lm)
            lm = lm.float()
            if lm.dim() == 4 and lm.shape[0] == 1:
                lm = lm.squeeze(0)
            if lm.dim() != 3:
                raise ValueError(f"Landmarks must be (T,N,D), got {tuple(lm.shape)} from key={key}")
            if lm.shape[-1] < 2:
                raise ValueError(f"Landmarks need at least x,y coordinates, got {tuple(lm.shape)}")
            return lm[..., :2].contiguous(), key
    if require:
        hint = f" in {path}" if path else ""
        raise KeyError(
            f"USE_LANDMARKS=True but no landmark key{hint}. "
            f"Expected one of {LANDMARK_KEYS}. Regenerate .pt files with lip/frame landmarks first."
        )
    return None, None


def infer_landmark_num_points(data_dir, pt_path=None):
    pt_path = pick_pt_file(data_dir, pt_path)
    data = torch.load(pt_path, map_location="cpu", weights_only=False)
    landmarks, key = find_landmarks_in_data(data, require=True, path=pt_path)
    print(f"Using landmarks from key='{key}' with {landmarks.shape[1]} points: {os.path.basename(pt_path)}")
    return int(landmarks.shape[1])



class MelTemporalUpsampleDecoder(nn.Module):
    """Upsample video-frame latents to HiFi-GAN mel-frame rate before decoding."""
    def __init__(self, base_decoder, sample_rate=16000, fps=25, hop_length=256):
        super().__init__()
        self.base_decoder = base_decoder
        self.sample_rate = sample_rate
        self.fps = fps
        self.hop_length = hop_length
        self.ratio = (sample_rate / fps) / hop_length

    def infer_target_len(self, video_len):
        return max(1, int(round(float(video_len) * self.ratio)))

    def forward(self, condition, target_len=None):
        if target_len is None:
            target_len = self.infer_target_len(condition.shape[1])
        if condition.shape[1] != target_len:
            condition = F.interpolate(
                condition.transpose(1, 2),
                size=int(target_len),
                mode="linear",
                align_corners=False,
            ).transpose(1, 2).contiguous()
        return self.base_decoder(condition)


class MelReconstructionLoss(nn.Module):
    def __init__(self, lambda_mel=1.0, lambda_delta=0.25, lambda_delta2=0.05, lambda_energy=0.05):
        super().__init__()
        self.lambda_mel = lambda_mel
        self.lambda_delta = lambda_delta
        self.lambda_delta2 = lambda_delta2
        self.lambda_energy = lambda_energy

    def _mask(self, x, lengths):
        T = x.shape[1]
        return (torch.arange(T, device=x.device).unsqueeze(0) < lengths.unsqueeze(1)).unsqueeze(-1).to(x.dtype)

    def _masked_l1(self, pred, target, lengths):
        mask = self._mask(pred, lengths)
        denom = mask.sum().clamp_min(1.0) * pred.shape[-1]
        return ((pred - target).abs() * mask).sum() / denom

    def _delta(self, x):
        return x[:, 1:] - x[:, :-1]

    def _energy(self, mel):
        return torch.logsumexp(mel.float(), dim=-1, keepdim=True)

    def forward(self, pred, target, lengths):
        pred = pred.float()
        target = target.float()
        lengths = lengths.to(device=pred.device, dtype=torch.long).clamp(min=1, max=pred.shape[1])
        loss = pred.new_tensor(0.0)
        if self.lambda_mel:
            loss = loss + self.lambda_mel * self._masked_l1(pred, target, lengths)
        if self.lambda_delta and pred.shape[1] > 1:
            loss = loss + self.lambda_delta * self._masked_l1(self._delta(pred), self._delta(target), (lengths - 1).clamp_min(1))
        if self.lambda_delta2 and pred.shape[1] > 2:
            loss = loss + self.lambda_delta2 * self._masked_l1(self._delta(self._delta(pred)), self._delta(self._delta(target)), (lengths - 2).clamp_min(1))
        if self.lambda_energy:
            loss = loss + self.lambda_energy * self._masked_l1(self._energy(pred), self._energy(target), lengths)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite mel loss: {float(loss.detach().cpu())}")
        return loss


class LandmarkEncoder(nn.Module):
    """Encode lip/face landmark positions and frame-to-frame velocity at video-frame rate."""
    def __init__(self, num_points, hidden_dim=256, out_dim=512, dropout=0.1):
        super().__init__()
        self.num_points = num_points
        in_dim = num_points * 4  # x, y, dx, dy
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.LayerNorm(out_dim),
        )

    def _normalize(self, landmarks):
        landmarks = torch.nan_to_num(landmarks.float(), nan=0.0, posinf=0.0, neginf=0.0)
        xy = landmarks[..., :2]
        center = xy.mean(dim=2, keepdim=True)
        xy = xy - center
        scale = xy.pow(2).sum(dim=-1).sqrt().amax(dim=2, keepdim=True).unsqueeze(-1).clamp_min(1e-4)
        return xy / scale

    def forward(self, landmarks):
        # landmarks: (B, T_video, N, 2)
        xy = self._normalize(landmarks)
        delta = xy[:, 1:] - xy[:, :-1]
        delta = torch.cat([torch.zeros_like(delta[:, :1]), delta], dim=1)
        x = torch.cat([xy, delta], dim=-1).flatten(start_dim=2)
        return self.net(x)


class VisualLandmarkEncoder(nn.Module):
    """Fuse the visual encoder with landmark dynamics before the mel decoder."""
    def __init__(self, visual_encoder, num_landmark_points, z_dim=512):
        super().__init__()
        self.visual_encoder = visual_encoder
        self.landmark_encoder = LandmarkEncoder(num_landmark_points, out_dim=z_dim)
        self.fusion = nn.Sequential(
            nn.Linear(z_dim * 2, z_dim),
            nn.LayerNorm(z_dim),
            nn.SiLU(),
            nn.Linear(z_dim, z_dim),
        )

    def forward(self, video, landmarks=None):
        if landmarks is None:
            raise ValueError("VisualLandmarkEncoder requires landmarks. Set USE_LANDMARKS=False for visual-only training.")
        z_video = self.visual_encoder(video)
        z_landmark = self.landmark_encoder(landmarks)
        if z_landmark.shape[1] != z_video.shape[1]:
            z_landmark = F.interpolate(
                z_landmark.transpose(1, 2),
                size=z_video.shape[1],
                mode="linear",
                align_corners=False,
            ).transpose(1, 2).contiguous()
        return self.fusion(torch.cat([z_video, z_landmark], dim=-1))


In [32]:
# ===== MULTI-RESOLUTION STFT LOSS =====
# Gi?p model h?c c?u tr?c t?n s? + pha ? nhi?u scale kh?c nhau
# Tham kh?o: Parallel WaveGAN, HiFi-GAN

import torch
import torch.nn as nn
import torch.nn.functional as F

class STFTLoss(nn.Module):
    """Single-resolution STFT loss computed in FP32 for numerical stability."""
    def __init__(self, fft_size=1024, hop_size=256, win_size=1024, eps=1e-5):
        super().__init__()
        self.fft_size = fft_size
        self.hop_size = hop_size
        self.win_size = win_size
        self.eps = eps
        self.register_buffer('window', torch.hann_window(win_size))

    def stft(self, x):
        # x: (B, T_samples) -> complex STFT. Keep STFT in FP32; FP16 log-magnitude can underflow to -inf.
        x = x.float()
        return torch.stft(
            x, self.fft_size, self.hop_size, self.win_size,
            self.window.to(device=x.device, dtype=torch.float32), return_complex=True
        )

    def forward(self, pred, target):
        # pred, target: (B, T_samples)
        pred = pred.float()
        target = target.float()
        pred_stft = self.stft(pred)
        target_stft = self.stft(target)
        pred_mag = pred_stft.abs().clamp_min(self.eps)
        target_mag = target_stft.abs().clamp_min(self.eps)
        denom = torch.norm(target_mag, p='fro').clamp_min(self.eps)
        sc_loss = torch.norm(target_mag - pred_mag, p='fro') / denom
        log_mag_loss = F.l1_loss(torch.log(pred_mag), torch.log(target_mag))
        loss = sc_loss + log_mag_loss
        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite STFTLoss: sc={float(sc_loss.detach().cpu())}, "
                f"log={float(log_mag_loss.detach().cpu())}"
            )
        return loss


class MultiResolutionSTFTLoss(nn.Module):
    """Multi-resolution STFT loss on full waveform tensors."""
    def __init__(self, fft_sizes=[256, 512, 1024], hop_sizes=[64, 128, 256], win_sizes=[256, 512, 1024]):
        super().__init__()
        self.stft_losses = nn.ModuleList([
            STFTLoss(fft, hop, win)
            for fft, hop, win in zip(fft_sizes, hop_sizes, win_sizes)
        ])

    def forward(self, pred, target):
        total = 0.0
        for stft_loss in self.stft_losses:
            total = total + stft_loss(pred, target)
        return total / len(self.stft_losses)


class CombinedAudioLoss(nn.Module):
    """MSE tr?n chunks + MR-STFT tr?n waveform, h? tr? padded batch b?ng lengths."""
    def __init__(self, lambda_mse=1.0, lambda_stft=1.0, hop=640):
        super().__init__()
        self.lambda_mse = lambda_mse
        self.lambda_stft = lambda_stft
        self.hop = hop
        self.mr_stft = MultiResolutionSTFTLoss(
            fft_sizes=[256, 512, 1024],
            hop_sizes=[64, 128, 256],
            win_sizes=[256, 512, 1024]
        )

    def _masked_mse(self, pred, target, lengths):
        B, T, C = pred.shape
        mask = torch.arange(T, device=pred.device).unsqueeze(0) < lengths.unsqueeze(1)
        mask = mask.unsqueeze(-1).to(dtype=pred.dtype)
        denom = mask.sum().clamp_min(1.0) * C
        return ((pred - target).pow(2) * mask).sum() / denom

    def _stft_loss_by_length(self, pred, target, lengths):
        losses = []
        for i, L in enumerate(lengths.tolist()):
            pred_wave = pred[i, :L].reshape(1, -1)
            target_wave = target[i, :L].reshape(1, -1)
            # STFT c?n t?i thi?u win_size; crop train hi?n ?? d?i, nh?ng guard ?? tr?nh clip qu? ng?n.
            if pred_wave.shape[1] < 1024:
                pad = 1024 - pred_wave.shape[1]
                pred_wave = F.pad(pred_wave, (0, pad))
                target_wave = F.pad(target_wave, (0, pad))
            losses.append(self.mr_stft(pred_wave, target_wave))
        return torch.stack(losses).mean()

    def forward(self, pred, target, lengths=None):
        if pred.ndim != 3 or target.ndim != 3:
            raise ValueError(f"CombinedAudioLoss expects (B,T,640), got pred={tuple(pred.shape)}, target={tuple(target.shape)}")
        if pred.shape != target.shape:
            raise ValueError(f"Shape mismatch: pred={tuple(pred.shape)}, target={tuple(target.shape)}")

        # Compute the loss in FP32 even when the model forward uses AMP/FP16.
        pred = pred.float()
        target = target.float()

        if lengths is None:
            lengths = torch.full((pred.shape[0],), pred.shape[1], device=pred.device, dtype=torch.long)
        else:
            lengths = lengths.to(device=pred.device, dtype=torch.long).clamp(min=1, max=pred.shape[1])

        loss = pred.new_tensor(0.0)
        if self.lambda_mse != 0:
            loss = loss + self.lambda_mse * self._masked_mse(pred, target, lengths)
        if self.lambda_stft != 0:
            loss = loss + self.lambda_stft * self._stft_loss_by_length(pred, target, lengths)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite CombinedAudioLoss: {float(loss.detach().cpu())}")
        return loss



def make_criterion(target_type, device):
    if target_type == "mel_hifigan":
        print("Loss function: MelReconstructionLoss (masked mel L1 + delta + energy)")
        return MelReconstructionLoss(lambda_mel=1.0, lambda_delta=0.25, lambda_delta2=0.05, lambda_energy=0.05).to(device)
    if target_type == "waveform":
        print("Loss function: CombinedAudioLoss (masked MSE + length-aware MR-STFT)")
        print("STFT resolutions: 256, 512, 1024")
        return CombinedAudioLoss(lambda_mse=1.0, lambda_stft=1.0).to(device)
    raise ValueError(f"Unsupported TARGET_TYPE={target_type}")


# DataLoader

In [33]:
from torch.utils.data import Dataset, DataLoader
import random

class VNLipDataset(Dataset):
    def __init__(self, data_dir, max_frames=None, random_crop=True, return_path=False,
                 target_type="mel_hifigan", use_landmarks=True):
        self.data_dir = data_dir
        self.max_frames = max_frames
        self.random_crop = random_crop
        self.return_path = return_path
        self.target_type = target_type
        self.use_landmarks = use_landmarks
        self.landmark_num_points = None
        if not os.path.isdir(self.data_dir):
            raise FileNotFoundError(f"Khong tim thay data_dir: {self.data_dir}")
        self.files = sorted(f for f in os.listdir(self.data_dir) if f.endswith('.pt'))
        if len(self.files) == 0:
            raise RuntimeError(f"Khong co file .pt trong {self.data_dir}")
        if self.use_landmarks:
            sample_path = os.path.join(self.data_dir, self.files[0])
            sample = torch.load(sample_path, map_location='cpu', weights_only=False)
            lm, _ = find_landmarks_in_data(sample, require=True, path=sample_path)
            self.landmark_num_points = int(lm.shape[1])

    def __len__(self):
        return len(self.files)

    def _get_target(self, data, file_path):
        if self.target_type == "mel_hifigan":
            if "mel" not in data:
                raise KeyError(f"{file_path} has no 'mel' target")
            target = data['mel'].float()
            n_mels = int(data.get('n_mels', 80))
            if target.dim() != 2:
                raise ValueError(f"Mel target must be 2D, got {tuple(target.shape)} in {file_path}")
            if target.shape[0] == n_mels and target.shape[1] != n_mels:
                target = target.transpose(0, 1).contiguous()
            target_len = int(data.get('mel_len', target.shape[0]))
            return target[:target_len], target_len

        if self.target_type == "waveform":
            if "audio" not in data:
                raise KeyError(f"{file_path} has no 'audio' target")
            target = data['audio'].float()
            target_len = int(data.get('audio_len', target.shape[0]))
            return target[:target_len], target_len

        raise ValueError(f"Unsupported target_type={self.target_type}")

    def _crop(self, video, target, target_len, landmarks=None, data=None):
        video_len = int(data.get('video_len', video.shape[1])) if data else video.shape[1]
        video_len = min(video_len, video.shape[1])
        target_len = min(int(target_len), target.shape[0])
        video = video[:, :video_len]
        target = target[:target_len]
        if landmarks is not None:
            landmarks = landmarks[:video_len]

        if self.max_frames is None or video_len <= self.max_frames:
            return video, target, landmarks

        if self.random_crop:
            start = random.randint(0, video_len - self.max_frames)
        else:
            start = (video_len - self.max_frames) // 2
        end = start + self.max_frames

        ratio = target_len / max(video_len, 1)
        target_start = int(round(start * ratio))
        target_end = int(round(end * ratio))
        target_end = max(target_start + 1, min(target_end, target_len))

        video = video[:, start:end]
        target = target[target_start:target_end]
        if landmarks is not None:
            landmarks = landmarks[start:end]
        return video, target, landmarks

    def __getitem__(self, idx):
        file_path = os.path.join(self.data_dir, self.files[idx])
        data = torch.load(file_path, map_location='cpu', weights_only=False)
        video = data['video'].float()
        target, target_len = self._get_target(data, file_path)

        if video.dim() == 3:
            video = video.unsqueeze(0)
        if video.dim() != 4 or target.dim() != 2:
            raise ValueError(f"Sai shape trong {file_path}: video={tuple(video.shape)}, target={tuple(target.shape)}")

        landmarks = None
        if self.use_landmarks:
            landmarks, _ = find_landmarks_in_data(data, require=True, path=file_path)

        video, target, landmarks = self._crop(video, target, target_len, landmarks=landmarks, data=data)

        if self.use_landmarks:
            if self.return_path:
                return video, landmarks, target, file_path
            return video, landmarks, target
        if self.return_path:
            return video, target, file_path
        return video, target


## Collate function (pad T)

In [34]:
def collate_pad(batch):
    has_path = isinstance(batch[0][-1], str)
    has_landmarks = (len(batch[0]) >= 3 and torch.is_tensor(batch[0][1]) and batch[0][1].dim() == 3)

    if has_landmarks and has_path:
        videos, landmarks, targets, paths = zip(*batch)
    elif has_landmarks:
        videos, landmarks, targets = zip(*batch)
        paths = None
    elif has_path:
        videos, targets, paths = zip(*batch)
        landmarks = None
    else:
        videos, targets = zip(*batch)
        landmarks = None
        paths = None

    video_lengths = torch.tensor([v.shape[1] for v in videos], dtype=torch.long)
    target_lengths = torch.tensor([t.shape[0] for t in targets], dtype=torch.long)
    T_video_max = int(video_lengths.max().item())
    T_target_max = int(target_lengths.max().item())

    padded_videos = []
    padded_targets = []
    padded_landmarks = []

    for i, (v, t) in enumerate(zip(videos, targets)):
        v_len = v.shape[1]
        t_len = t.shape[0]
        if v_len < T_video_max:
            pad_v = torch.zeros(v.shape[0], T_video_max - v_len, v.shape[2], v.shape[3], dtype=v.dtype)
            v = torch.cat([v, pad_v], dim=1)
        if t_len < T_target_max:
            pad_t = torch.zeros(T_target_max - t_len, t.shape[1], dtype=t.dtype)
            t = torch.cat([t, pad_t], dim=0)
        padded_videos.append(v)
        padded_targets.append(t)

        if has_landmarks:
            lm = landmarks[i]
            if lm.shape[0] < T_video_max:
                pad_lm = torch.zeros(T_video_max - lm.shape[0], lm.shape[1], lm.shape[2], dtype=lm.dtype)
                lm = torch.cat([lm, pad_lm], dim=0)
            padded_landmarks.append(lm)

    video_batches = torch.stack(padded_videos, dim=0)      # (B, 1, T_video, 112, 112)
    target_batches = torch.stack(padded_targets, dim=0)    # mel: (B, T_mel, 80), waveform: (B, T_video, 640)

    if has_landmarks:
        landmark_batches = torch.stack(padded_landmarks, dim=0)  # (B, T_video, N, 2)
        if has_path:
            return video_batches, landmark_batches, target_batches, target_lengths, list(paths)
        return video_batches, landmark_batches, target_batches, target_lengths

    if has_path:
        return video_batches, target_batches, target_lengths, list(paths)
    return video_batches, target_batches, target_lengths


## DataLoader


In [35]:
import os

# T4 16GB: bat dau voi 80 hoac 96. Tang dan neu VRAM con du.
MAX_FRAMES = 30
USE_LANDMARKS = True  # Mel experiment: set False for visual-only baseline.

DATA_DIR = MEL_DATA_DIR if TARGET_TYPE == "mel_hifigan" else WAVEFORM_DATA_DIR

dataset = VNLipDataset(
    DATA_DIR,
    max_frames=MAX_FRAMES,
    random_crop=True,
    target_type=TARGET_TYPE,
    use_landmarks=USE_LANDMARKS,
)

dataloader = DataLoader(
    dataset=dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_pad,
)

print(f"Dataset: {DATA_DIR} | target={TARGET_TYPE} | use_landmarks={USE_LANDMARKS} | files={len(dataset)}")


Dataset: Processed_Data_Mel_HiFiGAN | target=mel_hifigan | use_landmarks=True | files=2439


In [36]:
# ===== MODEL CONFIG =====
# TARGET_TYPE="mel_hifigan" makes decoder output (B, T_mel, 80) for HiFi-GAN.
# TARGET_TYPE="waveform" keeps the old direct waveform chunk output (B, T, 640).
ENCODER_TYPE = "snn"
DECODER_TYPE = "tfilm_siren"
TARGET_TYPE = globals().get("TARGET_TYPE", "mel_hifigan")
USE_LANDMARKS = globals().get("USE_LANDMARKS", False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
visual_encoder = build_encoder(ENCODER_TYPE).to(device)
if USE_LANDMARKS:
    num_landmark_points = getattr(dataset, "landmark_num_points", None) or infer_landmark_num_points(DATA_DIR)
    encoder = VisualLandmarkEncoder(visual_encoder, num_landmark_points=num_landmark_points).to(device)
else:
    encoder = visual_encoder

if TARGET_TYPE == "mel_hifigan":
    base_decoder = TFiLMSIRENDecoder(
        hidden_dim=256,
        out_dim=80,
        num_layers=4,
        use_conv=True,
        output_activation=None,
    ).to(device)
    decoder = MelTemporalUpsampleDecoder(base_decoder, sample_rate=16000, fps=25, hop_length=256).to(device)
    print(f"Encoder: {ENCODER_TYPE} | landmarks={USE_LANDMARKS} | Decoder: {DECODER_TYPE} -> mel_hifigan | Device: {device} | ratio={decoder.ratio:.2f}")
elif TARGET_TYPE == "waveform":
    decoder = TFiLMSIRENDecoder(out_dim=640, output_activation="tanh").to(device)
    print(f"Encoder: {ENCODER_TYPE} | landmarks={USE_LANDMARKS} | Decoder: {DECODER_TYPE} -> waveform | Device: {device}")
else:
    raise ValueError(f"Unsupported TARGET_TYPE={TARGET_TYPE}")


Encoder: snn | landmarks=True | Decoder: tfilm_siren -> mel_hifigan | Device: cuda | ratio=2.50


In [37]:

# ===== NON-SNN SMOKE TEST =====
# Ch?y cell n?y ?? ki?m tra nhanh encoder m?i tr??c khi train.
def smoke_test_encoder(encoder_type="non_snn", T=30, device=None):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    test_encoder = build_encoder(encoder_type).to(device).eval()
    test_decoder = TFiLMSIRENDecoder(out_dim=80).to(device).eval()
    dummy_video = torch.randn(2, 1, T, 112, 112, device=device)
    with torch.no_grad():
        z = test_encoder(dummy_video)
        audio_pred = test_decoder(z)
    print(f"{encoder_type} z:", z.shape, "finite:", torch.isfinite(z).all().item())
    print("audio_pred:", audio_pred.shape, "finite:", torch.isfinite(audio_pred).all().item())
    del test_encoder, test_decoder, dummy_video, z, audio_pred
    if device.type == 'cuda':
        torch.cuda.empty_cache()

smoke_test_encoder("non_snn", T=30, device=device)


c:\Users\Luc\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


non_snn z: torch.Size([2, 30, 512]) finite: True
audio_pred: torch.Size([2, 30, 80]) finite: True


In [38]:
def train_one_epoch(
        encoder: nn.Module,
        decoder: nn.Module,
        dataloader: DataLoader,
        optimizer: torch.optim.Optimizer,
        criterion: nn.Module,
        device: torch.device,
        use_mask: bool=True,
        max_grad_norm: Optional[float]=1.0,
        scaler: Optional[torch.amp.GradScaler]=None,
        log_memory_every: int=25,
    ):
    encoder.train()
    decoder.train()
    total_loss = 0.0
    processed_batches = 0
    amp_enabled = device.type == 'cuda'
    if scaler is None:
        scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)

    for batch_idx, batch in enumerate(tqdm(dataloader)):
        paths = None
        landmark_batch = None
        if len(batch) == 5:
            video_batch, landmark_batch, target_batch, lengths, paths = batch
        elif len(batch) == 4 and torch.is_tensor(batch[1]) and batch[1].dim() == 4:
            video_batch, landmark_batch, target_batch, lengths = batch
        elif len(batch) == 4:
            video_batch, target_batch, lengths, paths = batch
        else:
            video_batch, target_batch, lengths = batch
        lengths = lengths.to(device)

        video_batch = video_batch.to(device, non_blocking=True)
        target_batch = target_batch.to(device, non_blocking=True)
        if landmark_batch is not None:
            landmark_batch = landmark_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        functional.reset_net(encoder)

        try:
            with torch.amp.autocast('cuda', enabled=amp_enabled):
                if landmark_batch is not None:
                    z = encoder(video_batch, landmark_batch)
                else:
                    z = encoder(video_batch)
                if isinstance(criterion, MelReconstructionLoss):
                    target_pred = decoder(z, target_len=target_batch.shape[1])
                else:
                    target_pred = decoder(z)

            with torch.amp.autocast('cuda', enabled=False):
                if isinstance(criterion, (CombinedAudioLoss, MelReconstructionLoss)):
                    loss = criterion(target_pred.float(), target_batch.float(), lengths if use_mask else None)
                else:
                    loss = criterion(target_pred.float(), target_batch.float())
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite training loss at batch={batch_idx}: {float(loss.detach().cpu())}")

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            if max_grad_norm:
                torch.nn.utils.clip_grad_norm_(
                    list(encoder.parameters()) + list(decoder.parameters()),
                    max_grad_norm
                )

            scaler.step(optimizer)
            scaler.update()

            total_loss += float(loss.detach().cpu())
            processed_batches += 1

            if amp_enabled and log_memory_every and processed_batches % log_memory_every == 0:
                peak = torch.cuda.max_memory_allocated() / 1024**3
                reserved = torch.cuda.max_memory_reserved() / 1024**3
                print(f"  -> batch {processed_batches}: target_T={int(lengths.max().item())}, peak={peak:.2f}GB, reserved={reserved:.2f}GB")
                torch.cuda.reset_peak_memory_stats()

        except torch.cuda.OutOfMemoryError:
            file_hint = paths[0] if paths else '<unknown>'
            print(f"OOM tai batch={batch_idx}, target_T={int(lengths.max().item())}, file={file_hint}")
            raise
        finally:
            functional.reset_net(encoder)
            del video_batch, target_batch, lengths
            if landmark_batch is not None: del landmark_batch
            if 'z' in locals(): del z
            if 'target_pred' in locals(): del target_pred
            if 'loss' in locals(): del loss
            if device.type == 'cuda':
                torch.cuda.empty_cache()

    if processed_batches == 0:
        raise RuntimeError("Khong train duoc batch nao. Hay kiem tra dataset/max_frames/collate_fn.")

    return total_loss / processed_batches


In [39]:

def train_full(encoder, decoder, train_loader, optimizer, criterion, device,
               num_epochs=50, use_mask=True, max_grad_norm=1.0,
               checkpoint_dir="checkpoints", save_best=True, scaler=None):
    """Hu?n luy?n nhi?u epoch, in log loss, l?u checkpoint."""
    os.makedirs(checkpoint_dir, exist_ok=True)
    best_loss = float('inf')
    history = []
    if scaler is None:
        scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

    for epoch in range(1, num_epochs + 1):
        avg_loss = train_one_epoch(encoder, decoder, train_loader,
                                   optimizer, criterion, device,
                                   use_mask=use_mask,
                                   max_grad_norm=max_grad_norm,
                                   scaler=scaler)
        history.append(avg_loss)
        print(f"Epoch {epoch:3d}/{num_epochs} | Train Loss: {avg_loss:.6f}")

        state = {
            'epoch': epoch,
            'encoder_state_dict': encoder.state_dict(),
            'decoder_state_dict': decoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss': avg_loss,
            'max_frames': globals().get('MAX_FRAMES', None),
        }

        if epoch % 10 == 0 or epoch == num_epochs:
            checkpoint_path = os.path.join(checkpoint_dir, f"epoch_{epoch}.pth")
            torch.save(state, checkpoint_path)
            print(f"  -> ?? l?u checkpoint: {checkpoint_path}")

        if save_best and avg_loss < best_loss:
            best_loss = avg_loss
            best_path = os.path.join(checkpoint_dir, "best_model.pth")
            torch.save(state, best_path)
            print(f"  -> ?? l?u model t?t nh?t: {best_path}")

    print("Ho?n t?t hu?n luy?n!")
    return history


In [ ]:
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-4
)
criterion = make_criterion(TARGET_TYPE, device)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

avg_loss = train_one_epoch(
    encoder, decoder, dataloader, optimizer, criterion, device,
    use_mask=True, max_grad_norm=1.0, scaler=scaler
)
print(f"Loss {avg_loss}")


In [58]:
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-4
)
criterion = make_criterion(TARGET_TYPE, device)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
checkpoint_dir = f"my_checkpoints_{ENCODER_TYPE}_{TARGET_TYPE}_{'lm' if USE_LANDMARKS else 'visual'}"

history = train_full(encoder, decoder, dataloader, optimizer, criterion, device,
                      num_epochs=10, use_mask=True, max_grad_norm=1.0,
                      checkpoint_dir=checkpoint_dir, save_best=True,
                      scaler=scaler)


Loss function: MelReconstructionLoss (masked mel L1 + delta + energy)


  0%|          | 0/813 [00:00<?, ?it/s]

  -> batch 25: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 50: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 75: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 100: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 125: target_T=78, peak=3.31GB, reserved=3.47GB
  -> batch 150: target_T=75, peak=3.31GB, reserved=3.47GB
  -> batch 175: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 200: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 225: target_T=75, peak=3.31GB, reserved=3.47GB
  -> batch 250: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 275: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 300: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 325: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 350: target_T=75, peak=3.31GB, reserved=3.47GB
  -> batch 375: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 400: target_T=77, peak=3.31GB, reserved=3.47GB
  -> batch 425: target_T=78, peak=3.31GB, reserved=3.47GB
  -> batch 450: t

  0%|          | 0/813 [00:00<?, ?it/s]

  -> batch 25: target_T=75, peak=3.31GB, reserved=3.47GB
  -> batch 50: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 75: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 100: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 125: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 150: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 175: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 200: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 225: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 250: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 275: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 300: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 325: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 350: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 375: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 400: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 425: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 450: t

...

  0%|          | 0/813 [00:00<?, ?it/s]

  -> batch 25: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 50: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 75: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 100: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 125: target_T=77, peak=3.31GB, reserved=3.47GB
  -> batch 150: target_T=78, peak=3.31GB, reserved=3.47GB
  -> batch 175: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 200: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 225: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 250: target_T=75, peak=3.31GB, reserved=3.47GB
  -> batch 275: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 300: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 325: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 350: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 375: target_T=75, peak=3.31GB, reserved=3.47GB
  -> batch 400: target_T=76, peak=3.31GB, reserved=3.47GB
  -> batch 425: target_T=75, peak=3.31GB, reserved=3.47GB
  -> batch 450: t

  0%|          | 0/813 [00:00<?, ?it/s]

  -> batch 25: target_T=76, peak=3.31GB, reserved=3.47GB


KeyboardInterrupt: 

## 300 ep


In [55]:
# ===== OVERFIT 2 .PT FILES: VIDEO + LANDMARKS -> MEL =====
import os
import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from spikingjelly.activation_based import functional

# -----------------------------
# Config
# -----------------------------
OVERFIT_2PT_EPOCHS = 500
OVERFIT_2PT_LR = 3e-4
OVERFIT_2PT_MAX_FRAMES = 30
OVERFIT_2PT_BATCH_SIZE = 1
OVERFIT_2PT_SAVE_DIR = "overfit_2pt_checkpoints"
OVERFIT_2PT_SAVE_EVERY = 50

# Neu muon chon cu the 2 file, set list path o day.
# Neu None, cell se lay 2 file dau tien trong DATA_DIR.
OVERFIT_2PT_FILES = None
# OVERFIT_2PT_FILES = [
#     r"Processed_Data_Mel_HiFiGAN/xxx.pt",
#     r"Processed_Data_Mel_HiFiGAN/yyy.pt",
# ]


def build_overfit_2pt_dataset():
    data_dir = globals().get("DATA_DIR", globals().get("MEL_DATA_DIR", "Processed_Data_Mel_HiFiGAN"))

    if OVERFIT_2PT_FILES is None:
        base_dataset = VNLipDataset(
            data_dir,
            max_frames=OVERFIT_2PT_MAX_FRAMES,
            random_crop=True,
            target_type="mel_hifigan",
            use_landmarks=True,
            return_path=True,
        )
        if len(base_dataset) < 2:
            raise RuntimeError(f"Need at least 2 .pt files in {data_dir}, got {len(base_dataset)}")
        return Subset(base_dataset, [0, 1])

    class TwoFileDataset(torch.utils.data.Dataset):
        def __init__(self, files):
            self.files = files

        def __len__(self):
            return len(self.files)

        def __getitem__(self, idx):
            path = self.files[idx]
            data = torch.load(path, map_location="cpu", weights_only=False)

            video = data["video"].float()
            if video.dim() == 3:
                video = video.unsqueeze(0)

            mel = data["mel"].float()
            n_mels = int(data.get("n_mels", 80))
            if mel.dim() == 2 and mel.shape[0] == n_mels and mel.shape[1] != n_mels:
                mel = mel.transpose(0, 1).contiguous()

            landmarks, _ = find_landmarks_in_data(data, require=True, path=path)

            video_len = int(data.get("video_len", video.shape[1]))
            mel_len = int(data.get("mel_len", mel.shape[0]))

            video = video[:, :video_len]
            landmarks = landmarks[:video_len]
            mel = mel[:mel_len]

            if OVERFIT_2PT_MAX_FRAMES is not None and video.shape[1] > OVERFIT_2PT_MAX_FRAMES:
                start = 0
                end = start + OVERFIT_2PT_MAX_FRAMES

                ratio = mel.shape[0] / max(video.shape[1], 1)
                mel_start = int(round(start * ratio))
                mel_end = int(round(end * ratio))
                mel_end = max(mel_start + 1, min(mel_end, mel.shape[0]))

                video = video[:, start:end]
                landmarks = landmarks[start:end]
                mel = mel[mel_start:mel_end]

            return video, landmarks, mel, path

    return TwoFileDataset(OVERFIT_2PT_FILES)


def build_overfit_2pt_model(device):
    visual_encoder = build_encoder(globals().get("ENCODER_TYPE", "non_snn")).to(device)

    # Lay so diem landmarks tu dataset/file dau tien.
    if OVERFIT_2PT_FILES is not None:
        sample = torch.load(OVERFIT_2PT_FILES[0], map_location="cpu", weights_only=False)
        sample_lm, _ = find_landmarks_in_data(sample, require=True, path=OVERFIT_2PT_FILES[0])
        num_landmark_points = sample_lm.shape[1]
    else:
        num_landmark_points = getattr(dataset, "landmark_num_points", None)
        if num_landmark_points is None:
            num_landmark_points = infer_landmark_num_points(DATA_DIR)

    encoder = VisualLandmarkEncoder(
        visual_encoder,
        num_landmark_points=num_landmark_points,
    ).to(device)

    base_decoder = TFiLMSIRENDecoder(
        hidden_dim=512,
        out_dim=80,
        num_layers=4,
        use_conv=True,
        output_activation=None,
    ).to(device)

    decoder = MelTemporalUpsampleDecoder(
        base_decoder,
        sample_rate=16000,
        fps=25,
        hop_length=256,
    ).to(device)

    return encoder, decoder


# -----------------------------
# Build data/model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OVERFIT_2PT_SAVE_DIR, exist_ok=True)

overfit_dataset = build_overfit_2pt_dataset()
overfit_loader = DataLoader(
    overfit_dataset,
    batch_size=OVERFIT_2PT_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_pad,
)

encoder, decoder = build_overfit_2pt_model(device)

criterion = MelReconstructionLoss(
    lambda_mel=1.0,
    lambda_delta=0.0,
    lambda_delta2=0.0,
    lambda_energy=0.0,
).to(device)

optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=OVERFIT_2PT_LR,
)

scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
amp_enabled = device.type == "cuda"

print(f"Overfit dataset size: {len(overfit_dataset)}")
print(f"Device: {device}")
print(f"Epochs: {OVERFIT_2PT_EPOCHS}")
print(f"Save dir: {OVERFIT_2PT_SAVE_DIR}")


# -----------------------------
# Train
# -----------------------------
history = []

for epoch in range(1, OVERFIT_2PT_EPOCHS + 1):
    encoder.train()
    decoder.train()

    epoch_loss = 0.0
    steps = 0

    for batch in overfit_loader:
        paths = None
        if len(batch) == 5:
            video_batch, landmark_batch, mel_batch, mel_lengths, paths = batch
        elif len(batch) == 4 and torch.is_tensor(batch[1]) and batch[1].dim() == 4:
            video_batch, landmark_batch, mel_batch, mel_lengths = batch
        else:
            raise RuntimeError("Expected batch with landmarks: video, landmarks, mel, mel_lengths")

        video_batch = video_batch.to(device, non_blocking=True)
        landmark_batch = landmark_batch.to(device, non_blocking=True)
        mel_batch = mel_batch.to(device, non_blocking=True)
        mel_lengths = mel_lengths.to(device)

        optimizer.zero_grad(set_to_none=True)
        functional.reset_net(encoder)

        with torch.amp.autocast("cuda", enabled=amp_enabled):
            z = encoder(video_batch, landmark_batch)
            mel_pred = decoder(z, target_len=mel_batch.shape[1])

        with torch.amp.autocast("cuda", enabled=False):
            loss = criterion(mel_pred.float(), mel_batch.float(), mel_lengths)

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite loss at epoch={epoch}: {float(loss.detach().cpu())}")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(decoder.parameters()),
            1.0,
        )

        scaler.step(optimizer)
        scaler.update()

        epoch_loss += float(loss.detach().cpu())
        steps += 1

        del video_batch, landmark_batch, mel_batch, mel_lengths, z, mel_pred, loss
        functional.reset_net(encoder)
        if device.type == "cuda":
            torch.cuda.empty_cache()

    avg_loss = epoch_loss / max(steps, 1)
    history.append(avg_loss)

    if epoch == 1 or epoch % 10 == 0 or epoch == OVERFIT_2PT_EPOCHS:
        print(f"Epoch {epoch:3d}/{OVERFIT_2PT_EPOCHS} | loss={avg_loss:.6f}")

    if epoch % OVERFIT_2PT_SAVE_EVERY == 0 or epoch == OVERFIT_2PT_EPOCHS:
        ckpt_path = os.path.join(OVERFIT_2PT_SAVE_DIR, f"epoch_{epoch}.pth")
        torch.save({
            "epoch": epoch,
            "encoder_state_dict": encoder.state_dict(),
            "decoder_state_dict": decoder.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "loss": avg_loss,
            "history": history,
            "target_type": "mel_hifigan",
            "use_landmarks": True,
            "max_frames": OVERFIT_2PT_MAX_FRAMES,
        }, ckpt_path)
        print(f"  -> saved {ckpt_path}")

print("Done overfitting 2 .pt files.")

Overfit dataset size: 2
Device: cuda
Epochs: 500
Save dir: overfit_2pt_checkpoints
Epoch   1/500 | loss=4.720342
Epoch  10/500 | loss=1.807472
Epoch  20/500 | loss=0.840066
Epoch  30/500 | loss=0.803488
Epoch  40/500 | loss=0.858837
Epoch  50/500 | loss=0.905034
  -> saved overfit_2pt_checkpoints\epoch_50.pth
Epoch  60/500 | loss=1.740346
Epoch  70/500 | loss=0.820344
Epoch  80/500 | loss=0.806499
Epoch  90/500 | loss=0.865880
Epoch 100/500 | loss=0.744875
  -> saved overfit_2pt_checkpoints\epoch_100.pth
Epoch 110/500 | loss=0.757901
Epoch 120/500 | loss=0.846553
Epoch 130/500 | loss=0.828199
Epoch 140/500 | loss=0.870681
Epoch 150/500 | loss=0.821576
  -> saved overfit_2pt_checkpoints\epoch_150.pth
Epoch 160/500 | loss=0.887302
Epoch 170/500 | loss=0.930908
Epoch 180/500 | loss=0.999918
Epoch 190/500 | loss=1.019531
Epoch 200/500 | loss=0.831609
  -> saved overfit_2pt_checkpoints\epoch_200.pth
Epoch 210/500 | loss=0.890310
Epoch 220/500 | loss=0.937627
Epoch 230/500 | loss=0.824982
Ep

KeyboardInterrupt: 

## LOAD AND TEST

In [41]:
# ===== LOAD CHECKPOINT -> PREDICT MEL -> HIFIGAN WAV =====
import os
import sys
import types
import torch
import torchaudio
from spikingjelly.activation_based import functional

# -----------------------------
# 1. Fix SpeechBrain lazyload k2_fsa
# -----------------------------
# Xoa cac import SpeechBrain cu/loi trong kernel hien tai.
for module_name in list(sys.modules.keys()):
    if module_name == "speechbrain" or module_name.startswith("speechbrain."):
        del sys.modules[module_name]

# SpeechBrain 1.1.0 co lazy module k2_fsa, nhung train/decode HiFi-GAN khong can k2.
# Tao dummy module de import lazy khong fail.
sys.modules["speechbrain.integrations.k2_fsa"] = types.ModuleType(
    "speechbrain.integrations.k2_fsa"
)

from speechbrain.inference.vocoders import HIFIGAN
from speechbrain.utils.fetching import LocalStrategy


# -----------------------------
# 2. Config
# -----------------------------
CHECKPOINT_PATH = r"overfit_2pt_checkpoints\epoch_300.pth"
# Vi du neu muon set tay:
# CHECKPOINT_PATH = r"my_checkpoints_non_snn_mel_hifigan_lm\best_model.pth"

HIFIGAN_SOURCE = "speechbrain/tts-hifigan-libritts-16kHz"
HIFIGAN_SAVEDIR = "pretrained_models/tts-hifigan-libritts-16kHz"

PRED_WAV_OUT = "pred_from_checkpoint_hifigan.wav"
GT_WAV_OUT = "gt_from_checkpoint_hifigan.wav"


def find_checkpoint():
    candidates = []

    if CHECKPOINT_PATH is not None:
        candidates.append(CHECKPOINT_PATH)

    if "checkpoint_dir" in globals():
        candidates.append(os.path.join(checkpoint_dir, "best_model.pth"))

    enc = globals().get("ENCODER_TYPE", "non_snn")
    target = globals().get("TARGET_TYPE", "mel_hifigan")
    lm_tag = "lm" if globals().get("USE_LANDMARKS", True) else "visual"
    candidates.append(f"my_checkpoints_{enc}_{target}_{lm_tag}/best_model.pth")
    candidates.append(f"my_checkpoints_{enc}/best_model.pth")

    for path in candidates:
        if path and os.path.exists(path):
            return path

    raise FileNotFoundError(
        "Khong tim thay checkpoint. Hay set CHECKPOINT_PATH = r'.../best_model.pth'"
    )


def load_hifigan_cpu():
    return HIFIGAN.from_hparams(
        source=HIFIGAN_SOURCE,
        savedir=HIFIGAN_SAVEDIR,
        local_strategy=LocalStrategy.COPY_SKIP_CACHE,
        run_opts={"device": "cpu"},
    )


# -----------------------------
# 3. Rebuild model dung config train
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ENCODER_TYPE = globals().get("ENCODER_TYPE", "non_snn")
TARGET_TYPE = globals().get("TARGET_TYPE", "mel_hifigan")
USE_LANDMARKS = globals().get("USE_LANDMARKS", True)

visual_encoder = build_encoder(ENCODER_TYPE).to(device)

if USE_LANDMARKS:
    num_landmark_points = getattr(dataset, "landmark_num_points", None)
    if num_landmark_points is None:
        num_landmark_points = infer_landmark_num_points(DATA_DIR)
    encoder = VisualLandmarkEncoder(
        visual_encoder,
        num_landmark_points=num_landmark_points,
    ).to(device)
else:
    encoder = visual_encoder

base_decoder = TFiLMSIRENDecoder(
    hidden_dim=256,
    out_dim=80,
    num_layers=4,
    use_conv=True,
    output_activation=None,
).to(device)

decoder = MelTemporalUpsampleDecoder(
    base_decoder,
    sample_rate=16000,
    fps=25,
    hop_length=256,
).to(device)


# -----------------------------
# 4. Load checkpoint
# -----------------------------
ckpt_path = find_checkpoint()
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

encoder.load_state_dict(ckpt["encoder_state_dict"])
decoder.load_state_dict(ckpt["decoder_state_dict"])

encoder.eval()
decoder.eval()

print(f"Loaded checkpoint: {ckpt_path}")
print(f"Epoch: {ckpt.get('epoch')} | Loss: {ckpt.get('loss')}")


# -----------------------------
# 5. Predict mel tu 1 batch
# -----------------------------
batch = next(iter(dataloader))

landmark_batch = None
paths = None

if len(batch) == 5:
    video_batch, landmark_batch, mel_batch, mel_lengths, paths = batch
elif len(batch) == 4 and torch.is_tensor(batch[1]) and batch[1].dim() == 4:
    video_batch, landmark_batch, mel_batch, mel_lengths = batch
elif len(batch) == 4:
    video_batch, mel_batch, mel_lengths, paths = batch
else:
    video_batch, mel_batch, mel_lengths = batch

video_batch = video_batch.to(device)
mel_batch = mel_batch.to(device)
mel_lengths = mel_lengths.to(device)

if landmark_batch is not None:
    landmark_batch = landmark_batch.to(device)

with torch.no_grad():
    functional.reset_net(encoder)

    if landmark_batch is not None:
        z = encoder(video_batch, landmark_batch)
    else:
        z = encoder(video_batch)

    pred_mel = decoder(z, target_len=mel_batch.shape[1])

real_len = int(mel_lengths[0].item())

pred_specs = pred_mel[:1, :real_len].permute(0, 2, 1).detach().cpu()
gt_specs = mel_batch[:1, :real_len].permute(0, 2, 1).detach().cpu()

print(f"pred_mel: {tuple(pred_specs.shape)} | min={pred_specs.min():.3f}, max={pred_specs.max():.3f}")
print(f"gt_mel:   {tuple(gt_specs.shape)} | min={gt_specs.min():.3f}, max={gt_specs.max():.3f}")


# -----------------------------
# 6. Decode bang HiFi-GAN
# -----------------------------
hifi_gan = load_hifigan_cpu()

sr = 16000
hop = 256
mel_lens = torch.tensor([real_len], dtype=torch.long)

with torch.no_grad():
    pred_wav = hifi_gan.decode_batch(pred_specs, mel_lens=mel_lens, hop_len=hop)
    gt_wav = hifi_gan.decode_batch(gt_specs, mel_lens=mel_lens, hop_len=hop)

pred_wav = pred_wav.squeeze(1).cpu()
gt_wav = gt_wav.squeeze(1).cpu()

torchaudio.save(PRED_WAV_OUT, pred_wav, sr)
torchaudio.save(GT_WAV_OUT, gt_wav, sr)

print(f"Saved predicted wav: {PRED_WAV_OUT} | shape={tuple(pred_wav.shape)}")
print(f"Saved ground-truth wav: {GT_WAV_OUT} | shape={tuple(gt_wav.shape)}")

Loaded checkpoint: overfit_2pt_checkpoints\epoch_300.pth
Epoch: 300 | Loss: 0.5324792116880417


c:\Users\Luc\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using file found at 'c:\Users\Luc\OneDrive\Documents\Nhận dạng\pretrained_models\tts-hifigan-libritts-16kHz\hyperparams.yaml'
c:\Users\Luc\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
INFO:speechbrain.utils.fetching:Fetch generator.ckpt: Using file found at 'c:\Users\Luc\OneDrive\Documents\Nhận dạng\pretrained_models\tts-hifigan-libritts-16kHz\generator.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: generator


pred_mel: (1, 80, 74) | min=-7.665, max=-0.040
gt_mel:   (1, 80, 74) | min=-9.965, max=0.834
Saved predicted wav: pred_from_checkpoint_hifigan.wav | shape=(1, 21504)
Saved ground-truth wav: gt_from_checkpoint_hifigan.wav | shape=(1, 21504)
